# SQL 심화 — CTE와 윈도우 함수

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

# SQL 심화 — CTE와 윈도우 함수

## WITH (CTE) · OVER · PARTITION BY · 순위 함수 · 누적·이동 집계 · LAG·LEAD · ROLLUP

---

이전 시간 서브쿼리로 질문을 단계로 나눠봤습니다. 그런데 서브쿼리가 깊어지면 괄호가 겹겹이 쌓여 읽기 힘듭니다. 또 `GROUP BY`는 여러 행을 _하나로 접어버려서_, "각 행에 그 그룹의 순위·누적을 **나란히** 붙이는" 일은 못 했습니다.

오늘은 ① 복잡한 쿼리를 *이름 붙인 단계*로 푸는 **CTE(`WITH`)** 와, ② 행을 그대로 둔 채 그룹 계산을 붙이는 **윈도우 함수**를 배웁니다. SQL이 한 단계 강력해지는 날입니다.

> 💬 **오늘의 한 문장**  
> "CTE로 생각을 단계로 적고, 윈도우 함수로 행을 잃지 않고 비교한다."

## 📋 학습 목표

이 학습 노트북을 마치면 여러분은 다음을 할 수 있습니다.

1. `WITH` 절(CTE)로 복잡한 쿼리를 가독성 높은 단계로 나눠 작성할 수 있습니다.
2. 윈도우 함수가 `GROUP BY`와 무엇이 다른지 설명할 수 있습니다.
3. 순위(`ROW_NUMBER`·`RANK`·`DENSE_RANK`), 누적·이동(`SUM OVER`), 전·후 비교(`LAG`·`LEAD`)를 구현할 수 있습니다.
4. `PARTITION BY`로 그룹별 윈도우 계산을 수행할 수 있습니다.

_💡 위 목록을 천천히 읽고, 지금 할 수 있는 것과 아직 낯선 것을 마음속으로 표시해 봅시다. 노트북을 마친 뒤 다시 돌아와 비교하면 성장이 눈에 보일 것입니다._

## 📚 학습 목차

| Part          | 주제                                                       | 이 파트에서 답할 질문                          | 예상 시간            |
| ------------- | ---------------------------------------------------------- | ---------------------------------------------- | -------------------- |
| **0**         | 지난 시간 복습과 데이터 준비                               | 서브쿼리의 한계는 무엇이었습니까?              | 15분                 |
| **1**         | 공통 테이블 표현식 — `WITH` (CTE)                          | 겹겹의 서브쿼리를 어떻게 정리합니까?           | 45분                 |
| **2**         | 윈도우 함수 기초 — `OVER` · `PARTITION BY`                 | `GROUP BY`와 무엇이 다릅니까?                  | 20분                 |
| **3**         | 순위 함수 — `ROW_NUMBER`·`RANK`·`DENSE_RANK`               | 등수를 어떻게 매깁니까?                        | 35분                 |
| **4**         | 누적·이동 집계 — `SUM() OVER`와 프레임                     | 누적 합계와 이동 평균은?                       | 35분                 |
| **5**         | 이전·다음 행 비교 — `LAG` · `LEAD`                         | 전월 대비 증감은 어떻게?                       | 25분                 |
| **6**         | 값 참조와 구간 나누기 — `FIRST_VALUE`·`LAST_VALUE`·`NTILE` | 기준값 대비 비교, 등급 나누기는?               | 30분                 |
| **7**         | 소계·총계 — `ROLLUP` · `CUBE` · `GROUPING SETS`            | 합계 행을 SQL로 만들려면?                      | 35분                 |
| **연습 문제** | CTE와 윈도우 함수 5문항                                    | CTE·윈도우를 반복 훈련합니다                   | 30분                 |
| **종합 실습** | 월별 누적 매출과 전월 대비 증감률                          | CTE와 윈도우로 성장 추이를 분석할 수 있습니까? | 30분                 |
| **정리**      | 오늘의 퀴즈 · 개념 연결 도식 · 치트시트 · 과제             | 오늘 배운 것을 확인하고 정리합니다             | 20분                 |
|               | _합계_                                                     |                                                | _320분 (5시간 20분)_ |

_⏱️ 학습 속도는 사람마다 다릅니다 — 여러분의 속도로 가면 됩니다._

# Part 0. 오늘의 여정

## 🔁 지난 시간 복습

이전 시간에는 흩어진 테이블을 하나로 이었습니다.

- PK·FK — 테이블을 잇는 열쇠
- `INNER`/`LEFT JOIN` — 여러 테이블을 옆으로 합치기
- `UNION` — 두 결과를 세로로 이어 붙이기
- 서브쿼리 — 쿼리 안에 쿼리를 넣어 질문을 단계로

특히 마지막의 *서브쿼리*가 강력했지만, 단계가 늘면 괄호가 깊어져 읽기 어려웠습니다. 오늘은 그 단계를 **이름으로** 풀고, `GROUP BY`로는 못 했던 *순위·누적*을 다룹니다.

> ❓ **질문하기**  
> 서브쿼리와 `GROUP BY`의 불편함을, 무엇으로 넘어설 수 있습니까?

## ⚙️ 환경 설정과 실습 데이터

> ⚠️ **주의하기 — 셀은 위에서부터 순서대로 실행**  
> 코드 셀은 반드시 위에서부터 순서대로 실행합니다. 중간부터 실행하면 이전 셀의 변수를 찾지 못해 오류가 납니다.

이전과 동일한 네 테이블(`customers`·`orders`·`products`·`order_items`)을 사용합니다. 오늘은 _시간 흐름에 따른_ 분석이 많아 `orders`의 날짜를 자주 씁니다.

## 실습 환경 — LMS 또는 BigQuery

이 노트북의 SQL은 **LMS 실습 환경(DuckDB)** 또는 **BigQuery** 어느 쪽에서든 실행되도록 작성했습니다. 둘 중 **편한 쪽 하나만** 고르면 됩니다.

- **LMS에서 학습한다면** — 바로 아래 `[C1]` 준비 셀을 먼저 실행한 뒤, 이어지는 셀을 위에서부터 순서대로 실행합니다.
- **BigQuery에서 학습한다면** — `[C1]`은 건너뛰고, 각 셀의 SQL을 BigQuery 콘솔(또는 샌드박스)에 붙여 실행합니다. 테이블 이름의 `project_name.dataset_name` 부분만 여러분의 프로젝트·데이터셋 이름으로 바꾸면 됩니다.

> ⚠️ **주의하기 — 두 환경의 문법이 다른 곳은 딱 두 군데(날짜 함수)뿐입니다**  
> 해당 셀에는 `/* BigQuery: ... */` 형태로 두 표기를 나란히 적어 두었으니, 자기 환경에 맞는 쪽을 쓰면 됩니다. 전체 대조는 아래 표에서 확인합니다.

## 두 환경의 문법 차이 — 한눈에 보기

| 항목              | LMS (DuckDB)                   | BigQuery                   | 이 교안에서는                                                   |
| ----------------- | ------------------------------ | -------------------------- | --------------------------------------------------------------- |
| 테이블 이름       | `project_name.dataset_name.표` | 좌동                       | **공통 사용** — `[C1]` 준비 셀이 같은 이름 공간을 만들어 줍니다 |
| 문자열 타입       | `STRING`                       | `STRING`                   | 공통                                                            |
| 정수 타입         | `INT64`                        | `INT64`                    | 공통                                                            |
| 금액 타입         | `DECIMAL(12,2)`                | `DECIMAL(12,2)`            | 공통 (BigQuery는 `NUMERIC`으로 해석)                            |
| 합집합(중복 제거) | `UNION DISTINCT`               | `UNION DISTINCT`           | 공통 표기로 통일                                                |
| **월 단위 절단**  | `date_trunc('month', d)`       | `DATE_TRUNC(d, MONTH)`     | **함수명·인자 순서가 다름 → 병기**                              |
| **날짜 차이**     | `date_diff('day', 시작, 끝)`   | `DATE_DIFF(끝, 시작, DAY)` | **인자 순서가 반대 → 병기**                                     |

> 💡 **핵심짚기 — 나머지 문법은 전부 같습니다**  
> `SELECT`·`WHERE`·`GROUP BY`·`HAVING`·`JOIN`·`UNION`·서브쿼리·`CASE WHEN`·`COALESCE`·`NULLIF`·윈도우 함수·`ROLLUP`·`CREATE TABLE AS`까지, 이 모듈에서 배우는 거의 모든 문법이 두 환경에서 똑같이 동작합니다. **"SQL은 표준 언어"** 라는 말이 실감 나는 대목입니다.

> 📌 **실무 연결하기 — 왜 날짜 함수만 다를까요**  
> 날짜 처리는 SQL 표준이 늦게 정리된 영역이라 DB마다 독자 함수를 먼저 만들었습니다. 그래서 실무에서 DB를 옮길 때 **가장 먼저 깨지는 것도 날짜 함수**입니다. 지금 이 차이를 눈에 익혀 두면 나중에 크게 도움이 됩니다.

In [1]:
# 로컬 실행 준비 — DuckDB에 연결하고 %%sql 매직을 켭니다. (가장 먼저 한 번 실행)
import duckdb

%load_ext sql
%config SqlMagic.displaylimit = 50

conn = duckdb.connect()  # 메모리 DB. 파일로 남기려면 duckdb.connect("moodumall.duckdb")
%sql conn --alias duckdb

In [2]:
%%sql
-- [C1] LMS(DuckDB) 준비 — BigQuery로 학습한다면 이 셀은 건너뛰세요.
-- BigQuery와 똑같은 `project_name.dataset_name.표` 이름을 쓸 수 있도록 이름 공간을 만듭니다.
ATTACH IF NOT EXISTS ':memory:' AS project_name;
CREATE SCHEMA IF NOT EXISTS project_name.dataset_name;

Running query in 'duckdb'

Count


In [3]:
%%sql
-- [C2]
CREATE OR REPLACE TABLE project_name.dataset_name.customers (
  customer_id STRING,
  name STRING,
  country STRING,
  signup_date DATE,
  grade STRING
);

INSERT INTO project_name.dataset_name.customers VALUES
  ('C001', '김민준', 'Korea', '2023-01-05', 'Gold'),
  ('C002', '이서연', 'Korea', '2023-02-11', 'Silver'),
  ('C003', '박도윤', 'Japan', '2023-02-20', 'Bronze'),
  ('C004', '최지우', 'USA', '2023-03-03', 'Gold'),
  ('C005', '정하준', 'Korea', '2023-03-15', 'Silver'),
  ('C006', '강서윤', 'Korea', '2023-04-01', 'Bronze'),
  ('C007', '조은우', 'Japan', '2023-04-18', 'Silver'),
  ('C008', '윤지호', 'USA', '2023-05-09', 'Gold'),
  ('C009', '임하은', 'Korea', '2023-05-22', 'Bronze'),
  ('C010', '한예준', 'Korea', '2023-06-02', 'Silver'),
  ('C011', '오시우', NULL, '2023-06-19', 'Bronze'),
  ('C012', '신아린', 'Japan', '2023-07-07', 'Silver'),
  ('C013', '권준서', 'Korea', '2023-07-25', 'Gold'),
  ('C014', '황지안', 'USA', '2023-08-10', NULL),
  ('C015', '안수아', 'Korea', '2023-08-28', 'Bronze')

Running query in 'duckdb'

15 rows affected.

Count
15


In [4]:
%%sql
-- [C3]
CREATE OR REPLACE TABLE project_name.dataset_name.products (
  product_id STRING,
  product_name STRING,
  category STRING,
  price DECIMAL(12,2)
);

INSERT INTO project_name.dataset_name.products VALUES
  ('P01', '에어러너', 'Running', 89000),
  ('P02', '클래식 스니커즈', 'Sneakers', 65000),
  ('P03', '첼시 부츠', 'Boots', 145000),
  ('P04', '여름 샌들', 'Sandals', 38000),
  ('P05', '트레일 러너', 'Running', 119000),
  ('P06', '캔버스 스니커즈', 'Sneakers', 49000),
  ('P07', '워커 부츠', 'Boots', 175000),
  ('P08', '슬리퍼 샌들', 'Sandals', 25000),
  ('P09', '양말 세트', 'Accessory', 12000),
  ('P10', '운동화 끈', 'Accessory', 5000)

Running query in 'duckdb'

10 rows affected.

Count
10


In [5]:
%%sql
-- [C4]
CREATE OR REPLACE TABLE project_name.dataset_name.orders (
  order_id STRING,
  customer_id STRING,
  order_date DATE,
  status STRING,
  amount DECIMAL(12,2)
);

INSERT INTO project_name.dataset_name.orders VALUES
  ('O0001', 'C001', '2023-09-02', 'Paid', 125000),
  ('O0002', 'C002', '2023-09-05', 'Shipped', 89000),
  ('O0003', 'C001', '2023-09-11', 'Returned', 45000),
  ('O0004', 'C003', '2023-09-15', 'Paid', 230000),
  ('O0005', 'C004', '2023-09-20', 'Cancelled', NULL),
  ('O0006', 'C005', '2023-09-25', 'Shipped', 67000),
  ('O0007', 'C002', '2023-10-01', 'Paid', 158000),
  ('O0008', 'C006', '2023-10-04', 'Placed', 32000),
  ('O0009', 'C007', '2023-10-12', 'Shipped', 410000),
  ('O0010', 'C008', '2023-10-19', 'Paid', 99000),
  ('O0011', 'C001', '2023-10-23', 'Paid', 76000),
  ('O0012', 'C009', '2023-10-28', 'Cancelled', NULL),
  ('O0013', 'C010', '2023-11-02', 'Shipped', 142000),
  ('O0014', 'C004', '2023-11-08', 'Paid', 88000),
  ('O0015', 'C011', '2023-11-13', 'Placed', 53000),
  ('O0016', 'C012', '2023-11-19', 'Shipped', 175000),
  ('O0017', 'C002', '2023-11-24', 'Returned', 61000),
  ('O0018', 'C013', '2023-11-29', 'Paid', 320000),
  ('O0019', 'C005', '2023-12-03', 'Paid', 47000),
  ('O0020', 'C008', '2023-12-09', 'Shipped', 215000),
  ('O0021', 'C014', '2023-12-14', 'Placed', 38000),
  ('O0022', 'C001', '2023-12-20', 'Paid', 134000),
  ('O0023', 'C015', '2023-12-25', 'Shipped', 92000),
  ('O0024', 'C007', '2024-01-03', 'Paid', 268000),
  ('O0025', 'C010', '2024-01-09', 'Cancelled', NULL),
  ('O0026', 'C003', '2024-01-15', 'Paid', 119000),
  ('O0027', 'C013', '2024-01-22', 'Shipped', 405000),
  ('O0028', 'C006', '2024-01-28', 'Paid', 58000),
  ('O0029', 'C004', '2024-02-04', 'Returned', 73000),
  ('O0030', 'C012', '2024-02-11', 'Paid', 187000)

Running query in 'duckdb'

30 rows affected.

Count
30


In [6]:
%%sql
-- [C5]
CREATE OR REPLACE TABLE project_name.dataset_name.order_items (
  order_id STRING,
  product_id STRING,
  quantity INT64,
  unit_price DECIMAL(12,2)
);

INSERT INTO project_name.dataset_name.order_items VALUES
  ('O0001', 'P01', 1, 125000),
  ('O0002', 'P02', 2, 44500),
  ('O0003', 'P03', 1, 45000),
  ('O0004', 'P04', 2, 115000),
  ('O0006', 'P05', 1, 67000),
  ('O0007', 'P06', 2, 79000),
  ('O0008', 'P07', 1, 32000),
  ('O0009', 'P08', 2, 205000),
  ('O0010', 'P09', 1, 99000),
  ('O0011', 'P01', 2, 38000),
  ('O0013', 'P02', 1, 142000),
  ('O0014', 'P03', 2, 44000),
  ('O0015', 'P04', 1, 53000),
  ('O0016', 'P05', 2, 87500),
  ('O0017', 'P06', 1, 61000),
  ('O0018', 'P07', 2, 160000),
  ('O0019', 'P08', 1, 47000),
  ('O0020', 'P09', 2, 107500),
  ('O0021', 'P01', 1, 38000),
  ('O0022', 'P02', 2, 67000),
  ('O0023', 'P03', 1, 92000),
  ('O0024', 'P04', 2, 134000),
  ('O0026', 'P05', 1, 119000),
  ('O0027', 'P06', 2, 202500),
  ('O0028', 'P07', 1, 58000),
  ('O0029', 'P08', 2, 36500),
  ('O0030', 'P09', 1, 187000)

Running query in 'duckdb'

27 rows affected.

Count
27


# Part 1. 공통 테이블 표현식 — `WITH` (CTE)

이전 시간 `FROM`절 서브쿼리로 "고객별 총구매액을 먼저 구하고, 그중 20만 이상"을 풀었습니다. 그런데 단계가 늘면 괄호 속 괄호가 되어 읽기 어렵습니다. **CTE**는 그 단계에 *이름*을 붙여 위에서 아래로 읽히게 합니다.

> ❓ **질문하기**  
> 겹겹이 쌓인 서브쿼리를, 어떻게 사람이 읽기 좋게 풀어냅니까?

## 🧩 개념 잡기

CTE(Common Table Expression)는 **"임시 이름표를 붙인 중간 결과"** 입니다. 요리로 치면 "① 재료 손질 → ② 볶기 → ③ 플레이팅"처럼 단계마다 이름을 붙여 두고, 다음 단계에서 그 이름을 쓰는 겁니다.

```sql
WITH 단계이름 AS (
    SELECT ...          -- 중간 결과에 이름 붙이기
)
SELECT ... FROM 단계이름   -- 그 이름을 테이블처럼 사용
```

서브쿼리를 _밖으로 꺼내_ 이름을 붙인 것과 같습니다. 깊은 괄호 → 위에서 아래로 읽는 단계로 바뀝니다.

▶️ **코드 실행하기 · 코드 셀 6 [C6]**  
— 고객별 총 구매액을 **CTE(`WITH`)로 이름 붙여** 만든 뒤, **총 구매액이 20만원 이상인 고객만** 큰 순으로 조회합니다.

In [7]:
%%sql
-- [C6]
WITH customer_total AS (
  SELECT
    customer_id,
    SUM(amount) AS total_amount
  FROM project_name.dataset_name.orders
  WHERE
    amount IS NOT NULL
  GROUP BY
    customer_id
)
SELECT
  *
FROM customer_total
WHERE
  total_amount >= 200000
ORDER BY
  total_amount DESC

Running query in 'duckdb'

customer_id,total_amount
C013,725000.00
C007,678000.00
C001,380000.00
C012,362000.00
C003,349000.00
C008,314000.00
C002,308000.00


> 👀 **결과읽기 — [C6]**  
> `WITH customer_total AS (...)` 가 "고객별 총구매액"이라는 중간 표를 만들고, 아래 `SELECT`가 그 표를 _테이블처럼_ 씁니다. 이전 시간의 `FROM (서브쿼리) AS t`와 결과는 같지만 훨씬 읽기 좋습니다.

> 🐼 **pandas와 비교**  
> CTE는 pandas에서 중간 결과를 변수에 담는 것과 같습니다. `customer_total = df.groupby('customer_id')['amount'].sum().reset_index()` 후 `customer_total[customer_total.total>=200000]`.

▶️ **코드 실행하기 · 코드 셀 7 [C7]**  
— **CTE를 두 단계로 이어**(집계 → 고객명 결합), 총 구매액 상위 5건을 이름·등급과 함께 조회합니다.

In [8]:
%%sql
-- [C7]
WITH customer_total AS (
  SELECT
    customer_id,
    SUM(amount) AS total_amount
  FROM project_name.dataset_name.orders
  WHERE
    amount IS NOT NULL
  GROUP BY
    customer_id
), named AS (
  SELECT
    c.name,
    c.grade,
    t.total_amount
  FROM customer_total AS t
  JOIN project_name.dataset_name.customers AS c
    ON t.customer_id = c.customer_id
)
SELECT
  *
FROM named
ORDER BY
  total_amount DESC
LIMIT 5

Running query in 'duckdb'

name,grade,total_amount
권준서,Gold,725000.00
조은우,Silver,678000.00
김민준,Gold,380000.00
신아린,Silver,362000.00
박도윤,Bronze,349000.00


> 👀 **결과읽기 — [C7]**  
> 첫 CTE(`customer_total`)가 총구매액을, 둘째 CTE(`named`)가 거기에 고객 이름을 붙입니다. 단계를 _순서대로 읽으면_ 됩니다. 복잡한 분석일수록 CTE의 가치가 커집니다.

> 📌 **실무 연결하기 — 여러 단계를 거치는 분석**  
> CTE는 _단계가 많은 지표 계산_(금융 리스크 점수), _전처리→집계→필터 파이프라인_(마케팅 세그먼트) 등 "여러 단계를 거치는 분석"의 표준 작성법입니다.

## ⌨️ 백문이 불여일타 (1)

```
[문제]
1) CTE를 사용해, 상태별 매출 합을 구하는 단계(status_revenue)를 만듭니다. (금액이 있는 주문만)
2) 그중 매출이 50만원 이상인 상태만 조회합니다.
```

In [9]:
%%sql
-- [C8] ⌨️ 백문이 불여일타 (1) — 상태별 매출 CTE + 50만원 이상 필터

WITH status_revenue AS (
    SELECT status, SUM(amount) AS revenue
    FROM project_name.dataset_name.orders
    WHERE amount IS NOT NULL
    GROUP BY status
)
SELECT * FROM status_revenue
WHERE revenue >= 500000

Running query in 'duckdb'

status,revenue
Paid,1909000.00
Shipped,1595000.00


<details>
<summary>(펼치기) 🔑 힌트</summary>

- `WITH status_revenue AS ( ... )` 형태로 중간 단계에 이름을 붙이는 것부터 시작합니다.
- CTE 안에서는 `GROUP BY status`와 `SUM(amount)`으로 상태별 매출을 구합니다. 금액이 있는 주문만 남기려면 `WHERE amount IS NOT NULL`이 필요합니다.
- 바깥 `SELECT`에서는 `status_revenue`를 보통 테이블처럼 조회하면서 `WHERE revenue >= 500000`으로 거르면 됩니다.

</details>

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
WITH status_revenue AS (
  SELECT
    status,
    SUM(amount) AS revenue
  FROM project_name.dataset_name.orders
  WHERE
    amount IS NOT NULL
  GROUP BY
    status
)
SELECT
  *
FROM status_revenue
WHERE
  revenue >= 500000
ORDER BY
  revenue DESC
```

</details>

## ⌨️ 백문이 불여일타 (2)

```
[문제]
1) order_items와 products를 JOIN해 카테고리별 매출을 구하는 CTE(cat_rev)를 만듭니다.
2) 그 결과에서 매출이 전체 카테고리 평균보다 높은 카테고리만 조회합니다.
```

In [10]:
%%sql
-- [C9] ⌨️ 백문이 불여일타 (2) — 2단계 CTE로 평균 이상 카테고리 찾기

WITH cat_rev AS (
    SELECT p.category, SUM(oi.quantity * oi.unit_price) AS revenue
    FROM project_name.dataset_name.order_items as oi
    JOIN project_name.dataset_name.products as p
      ON oi.product_id = p.product_id
    GROUP BY p.category
)
SELECT * FROM cat_rev
WHERE revenue > (
  SELECT AVG(revenue) FROM cat_rev
)

Running query in 'duckdb'

category,revenue
Sneakers,989000.00
Sandals,1081000.00


<details>
<summary>(펼치기) 🔑 힌트</summary>

- CTE 안에서 `order_items`와 `products`를 `JOIN`하고 `p.category`로 `GROUP BY`합니다. 매출은 `SUM(oi.quantity * oi.unit_price)`로 계산합니다.
- "전체 평균보다 높은"은 바깥 `SELECT`의 `WHERE`에서 `revenue > (SELECT AVG(revenue) FROM cat_rev)`처럼 CTE를 서브쿼리로 한 번 더 참조하면 됩니다.

</details>

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
WITH cat_rev AS (
  SELECT
    p.category,
    SUM(oi.quantity * oi.unit_price) AS revenue
  FROM project_name.dataset_name.order_items AS oi
  JOIN project_name.dataset_name.products AS p
    ON oi.product_id = p.product_id
  GROUP BY
    p.category
)
SELECT
  *
FROM cat_rev
WHERE
  revenue > (
    SELECT
      AVG(revenue)
    FROM cat_rev
  )
ORDER BY
  revenue DESC
```

</details>

## 🚦 짚고 넘어가기

다음 질문에 답할 수 있으면 다음 Part로 넘어가세요. 틀려도 괜찮습니다.

1. CTE(`WITH`)는 무엇을 합니까?
2. `FROM`절 서브쿼리와 CTE는 어떤 관계입니까?
3. CTE를 여러 개 이어 쓸 수 있습니까?

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

1. 쿼리의 중간 결과에 이름을 붙여, 이어지는 쿼리에서 보통 테이블처럼 참조할 수 있게 해줍니다. 복잡한 쿼리를 "단계 1 → 단계 2 → 최종" 순서로 읽히게 정리해줍니다.
2. 하는 일은 같습니다. `FROM`절 서브쿼리를 밖으로 꺼내 이름을 붙인 것이 CTE라서 서로 바꿔 쓸 수 있고, 단계가 많아질수록 CTE 쪽이 읽기 좋습니다.
3. 네. `WITH a AS (...), b AS (...)`처럼 쉼표로 이어 쓸 수 있고, 뒤의 CTE가 앞의 CTE를 참조할 수도 있습니다.

</details>

> ⏭️ **다음 학습 예고**  
> CTE로 단계를 정리했습니다. 그런데 "각 주문 옆에 *그 고객의 누적 구매액*을 나란히" 같은 건 `GROUP BY`로는 안 됩니다. 행을 잃지 않는 **윈도우 함수**가 필요합니다.

# Part 2. 윈도우 함수 기초 — `OVER` · `PARTITION BY`

`GROUP BY`는 여러 행을 _하나로 접습니다_. 그래서 "상태별 매출 합"은 상태마다 한 줄이 됐습니다. 하지만 "각 주문을 그대로 두고, 옆에 *그 상태의 합계*나 *순위*를 붙이고" 싶을 때가 있습니다. 이때 **윈도우 함수**를 씁니다.

> ❓ **질문하기**  
> `GROUP BY`처럼 그룹 계산을 하되, _행은 그대로 유지_ 하려면?

## 🧩 개념 잡기

```
GROUP BY: 행을 접는다              윈도우: 행을 유지하고 옆에 붙인다
 status  합계                      order_id  status  amount  상태별합계
 Paid    1,500,000                  O0001    Paid    125000   1,500,000
 (3줄로 요약)                       O0007    Paid    158000   1,500,000
                                    ... (원래 행 수 그대로)
```

핵심은 `OVER`입니다. 집계함수 뒤에 `OVER (...)`를 붙이면, 행을 접지 않고 _그 행이 속한 창(Window)_ 의 계산을 옆에 붙여줍니다.

```sql
SUM(amount) OVER (PARTITION BY status) /* 상태별 합계를 각 행에 붙임 */
```

## `OVER`와 `PARTITION BY`

- `OVER ()` : 전체를 하나의 창으로
- `OVER (PARTITION BY 열)` : 그 열의 값별로 창을 나눠 계산 (`GROUP BY`의 '그룹'과 비슷한 역할이되 행은 유지)

> ⚠️ **주의하기 — 차이 요약**  
> `GROUP BY`는 _결과 행 수를 줄이고_, 윈도우 함수는 _행 수를 유지_ 합니다. `OVER (ORDER BY ...)`의 정렬은 윈도우 함수가 계산할 순서를 정할 뿐, 최종 출력 순서를 보장하지 않습니다. 화면에 보이는 행 순서는 쿼리 마지막의 `ORDER BY`로 따로 지정합니다.

▶️ **코드 실행하기 · 코드 셀 10 [C10]**  
— **금액이 입력된** 주문마다 **상태별 총매출과 전체 총매출을 같은 행에 붙입니다** — `GROUP BY`와 달리 행 수가 줄지 않습니다.

In [11]:
%%sql
-- [C10]
SELECT
  order_id,
  status,
  amount,
  SUM(amount) OVER (PARTITION BY status) AS 상태별_총매출,
  SUM(amount) OVER () AS 전체_총매출
FROM project_name.dataset_name.orders
WHERE
  amount IS NOT NULL
ORDER BY
  status NULLS LAST,
  amount DESC

Running query in 'duckdb'

order_id,status,amount,상태별_총매출,전체_총매출
O0018,Paid,320000.00,1909000.00,3806000.00
O0024,Paid,268000.00,1909000.00,3806000.00
O0004,Paid,230000.00,1909000.00,3806000.00
O0030,Paid,187000.00,1909000.00,3806000.00
O0007,Paid,158000.00,1909000.00,3806000.00
O0022,Paid,134000.00,1909000.00,3806000.00
O0001,Paid,125000.00,1909000.00,3806000.00
O0026,Paid,119000.00,1909000.00,3806000.00
O0010,Paid,99000.00,1909000.00,3806000.00
O0014,Paid,88000.00,1909000.00,3806000.00


> 👀 **결과읽기 — [C10]**  
> 행은 27건 그대로인데, 각 행 옆에 *그 상태의 합계*와 *전체 합계*가 붙었습니다. `GROUP BY`였다면 상태별 3~4줄로 접혔을 겁니다. "개별 + 그룹 요약"을 한 화면에 보는 게 윈도우 함수의 힘입니다.

> 📌 **실무 연결하기 — 개별과 전체를 함께 보기**  
> "각 거래 + 그 계좌의 누적"(금융), "각 측정값 + 그 라인의 평균"(제조)처럼 _개별과 전체를 함께_ 봐야 하는 곳에 윈도우 함수가 쓰입니다.

> 🐼 **pandas와 비교**  
> `SUM(x) OVER (PARTITION BY g)` 는 pandas의 `df.groupby('g')['x'].transform('sum')` 과 같습니다. `groupby().sum()`(행을 접음)과 `groupby().transform('sum')`(행 유지) 의 차이가, 곧 `GROUP BY`와 윈도우 함수의 차입니다.

## 🚦 짚고 넘어가기

1. `GROUP BY`와 윈도우 함수의 가장 큰 차이는 (행 수 측면에서) 무엇입니까?
2. `OVER (PARTITION BY status)` 는 무엇을 합니까?
3. `OVER ()` (괄호 비움)는 어떤 범위입니까?

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

1. `GROUP BY`는 그룹당 한 행으로 행을 줄이지만, 윈도우 함수는 행 수를 그대로 유지한 채 집계 결과를 각 행 옆에 붙입니다.
2. 행 전체를 `status` 값별 그룹(창)으로 나눠, 각 행이 자기가 속한 그룹 안에서만 집계되게 합니다.
3. 전체 행이 하나의 창이 됩니다. 전체 합계·전체 평균을 모든 행 옆에 똑같이 붙일 때 씁니다.

</details>

> ⏭️ **다음 학습 예고**  
> 윈도우의 기본을 알았으니, 가장 많이 쓰는 윈도우 함수 — **순위 매기기** 부터 배웁니다.

# Part 3. 순위 함수 — `ROW_NUMBER` · `RANK` · `DENSE_RANK`

"매출 1등 고객", "카테고리별 베스트셀러 3위까지" — 순위는 분석에서 가장 자주 쓰는 윈도우 함수입니다. 세 가지 순위 함수의 *미묘한 차이*를 익힙니다.

> ❓ **질문하기**  
> 등수를 어떻게 매기고, 동점은 어떻게 처리합니까?

## 🧩 개념 잡기

세 함수는 **동점 처리 방식**이 다릅니다.

| 함수           | 동점 처리                            | 예 (100, 100, 90) |
| -------------- | ------------------------------------ | ----------------- |
| `ROW_NUMBER()` | 무조건 1,2,3 (동점도 임의 순서)      | 1, 2, 3           |
| `RANK()`       | 동점은 같은 등수, 다음은 _건너뜀_    | 1, 1, 3           |
| `DENSE_RANK()` | 동점은 같은 등수, 다음은 _안 건너뜀_ | 1, 1, 2           |

```sql
RANK() OVER (ORDER BY amount DESC) /* 금액 큰 순으로 등수 */
```

`OVER (ORDER BY ...)` 의 정렬 기준으로 등수가 매겨집니다.

> ⚠️ **주의하기 — `ROW_NUMBER`의 동점 순서는 추가 기준이 필요합니다**  
> 금액이 같은 행에 `ROW_NUMBER()`를 사용하면 어느 행이 먼저 번호를 받을지 정해지지 않습니다. `ORDER BY amount DESC, order_id`처럼 유일한 두 번째 기준을 추가하면 결과를 재현할 수 있습니다. 반면 공동 등수를 유지하려는 `RANK`·`DENSE_RANK`에는 등수 기준인 금액만 사용합니다.

▶️ **코드 실행하기 · 코드 셀 11 [C11]**  
— 금액이 큰 순으로 **`ROW_NUMBER`·`RANK`·`DENSE_RANK` 세 순위를 나란히** 매겨 상위 8건에서 차이를 비교합니다.

In [12]:
%%sql
-- [C11]
SELECT
  order_id,
  amount,
  ROW_NUMBER() OVER (ORDER BY amount DESC, order_id) AS row_number,
  RANK() OVER (ORDER BY amount DESC) AS rank,
  DENSE_RANK() OVER (ORDER BY amount DESC) AS dense_rank
FROM project_name.dataset_name.orders
WHERE
  amount IS NOT NULL
ORDER BY
  amount DESC
LIMIT 8

Running query in 'duckdb'

order_id,amount,row_number,rank,dense_rank
O0009,410000.00,1,1,1
O0027,405000.00,2,2,2
O0018,320000.00,3,3,3
O0024,268000.00,4,4,4
O0004,230000.00,5,5,5
O0020,215000.00,6,6,6
O0030,187000.00,7,7,7
O0016,175000.00,8,8,8


> 👀 **결과읽기 — [C11]**  
> 세 열의 숫자가 어떻게 다른지 확인합니다(동점이 있다면 차이가 드러납니다). 보통 *유일한 일련번호*가 필요하면 `ROW_NUMBER`, *공동 등수*가 필요하면 `RANK`/`DENSE_RANK`를 씁니다.

▶️ **코드 실행하기 · 코드 셀 12 [C12]**  
— 상품별 매출을 구한 뒤, **카테고리 안에서의 순위**를 매깁니다(`PARTITION BY`).

In [13]:
%%sql
-- [C12]
WITH product_sales AS (
  SELECT
    p.category,
    p.product_name,
    SUM(oi.quantity * oi.unit_price) AS revenue
  FROM project_name.dataset_name.order_items AS oi
  JOIN project_name.dataset_name.products AS p
    ON oi.product_id = p.product_id
  GROUP BY
    p.category,
    p.product_name
)
SELECT
  category,
  product_name,
  revenue,
  RANK() OVER (PARTITION BY category ORDER BY revenue DESC) AS 카테고리내_순위
FROM product_sales
ORDER BY
  category NULLS LAST,
  카테고리내_순위 NULLS LAST

Running query in 'duckdb'

category,product_name,revenue,카테고리내_순위
Accessory,양말 세트,501000.00,1
Boots,워커 부츠,410000.00,1
Boots,첼시 부츠,225000.00,2
Running,트레일 러너,361000.00,1
Running,에어러너,239000.00,2
Sandals,여름 샌들,551000.00,1
Sandals,슬리퍼 샌들,530000.00,2
Sneakers,캔버스 스니커즈,624000.00,1
Sneakers,클래식 스니커즈,365000.00,2


> 👀 **결과읽기 — [C12]**  
> `PARTITION BY category`로 _카테고리마다 순위를 따로_ 매겼습니다. 각 카테고리에서 1위가 다시 시작됩니다. "그룹별 TOP N"의 핵심 패턴입니다.

> 📌 **실무 연결하기 — 그룹 안에서의 순위**  
> _지점별 매출 순위_(유통), _부서별 성과 등수_(인사), _카테고리별 인기 상품_(이커머스) 등 "그룹 안에서의 순위"에 항상 쓰입니다.

## ⌨️ 백문이 불여일타 (3)

```
[문제] 고객별 총구매액(금액 있는 것만)을 구해, RANK()로 구매액 순위를 매기고 구매액이 큰 고객 5행을 보여주세요. (CTE + 윈도우)
```

In [14]:
%%sql
-- [C13] ⌨️ 백문이 불여일타 (3) — 고객별 총구매액 RANK 상위 5명

WITH t AS (
  SELECT
    customer_id,
    SUM(amount) AS total
  FROM project_name.dataset_name.orders
  WHERE amount IS NOT NULL
  GROUP BY customer_id
), ranked AS (
  SELECT 
    customer_id, 
    total,
    RANK() OVER (ORDER BY total DESC) AS 순위
  FROM t
)
SELECT * FROM ranked
WHERE 순위 <= 5
ORDER BY
  순위,
  customer_id

Running query in 'duckdb'

customer_id,total,순위
C013,725000.00,1
C007,678000.00,2
C001,380000.00,3
C012,362000.00,4
C003,349000.00,5


<details>
<summary>(펼치기) 🔑 힌트</summary>

- 1단계 CTE에서 `GROUP BY customer_id`와 `SUM(amount)`으로 고객별 총구매액을 구합니다.
- 바깥 `SELECT`에서 `RANK() OVER (ORDER BY total DESC)`로 순위를 매깁니다 — 행을 줄이지 않고 순위 열이 옆에 붙습니다.
- 결과를 5행으로 제한하려면 순위와 고객번호로 정렬한 뒤 `LIMIT 5`를 붙입니다. `RANK`의 5위까지 공동 순위자를 모두 포함하는 문제와는 구분합니다.

</details>

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
WITH t AS (
  SELECT
    customer_id,
    SUM(amount) AS total
  FROM project_name.dataset_name.orders
  WHERE
    amount IS NOT NULL
  GROUP BY
    customer_id
)
SELECT
  customer_id,
  total,
  RANK() OVER (ORDER BY total DESC) AS 순위
FROM t
ORDER BY
  순위 NULLS LAST,
  customer_id
LIMIT 5
```

**확인할 점**

- **5행과 5위까지는 다릅니다** — 위 쿼리는 결과를 5행으로 자릅니다. 공동 5위까지 모두 보여주려면 순위를 CTE로 한 번 더 감싼 뒤 `WHERE 순위 <= 5`로 거릅니다.

</details>

## 실전 패턴 — 그룹별 '1등만' 뽑기

순위 함수의 가장 강력한 쓰임은 **"각 그룹의 1등만 뽑기"** 입니다. `ROW_NUMBER`로 그룹별 순위를 매긴 뒤, _서브쿼리(또는 CTE)로 감싸 1등만_ 거릅니다. (윈도우 함수 결과는 `WHERE`에서 바로 못 거르므로 한 번 감싸는 게 표준 패턴입니다.)

▶️ **코드 실행하기 · 코드 셀 14 [C14]**  
— 카테고리별 순위를 매긴 결과를 CTE로 감싸, **각 카테고리의 1등 상품만** 남깁니다.

In [15]:
%%sql
-- [C14]
WITH ranked AS (
  SELECT
    p.category,
    p.product_name,
    SUM(oi.quantity * oi.unit_price) AS revenue,
    ROW_NUMBER() OVER (PARTITION BY p.category ORDER BY SUM(oi.quantity * oi.unit_price) DESC, p.product_name) AS rn
  FROM project_name.dataset_name.order_items AS oi
  JOIN project_name.dataset_name.products AS p
    ON oi.product_id = p.product_id
  GROUP BY
    p.category,
    p.product_name
)
SELECT
  category,
  product_name,
  revenue
FROM ranked
WHERE
  rn = 1
ORDER BY
  revenue DESC

Running query in 'duckdb'

category,product_name,revenue
Sneakers,캔버스 스니커즈,624000.00
Sandals,여름 샌들,551000.00
Accessory,양말 세트,501000.00
Boots,워커 부츠,410000.00
Running,트레일 러너,361000.00


> 👀 **결과읽기 — [C14]**  
> 안쪽 CTE에서 카테고리별 순위(`rn`)를 매기고, 바깥에서 `WHERE rn = 1`로 각 카테고리에서 **한 상품만** 남겼습니다. 매출이 같을 때는 상품명 순으로 하나를 고릅니다. 공동 1위를 모두 남기려면 `ROW_NUMBER` 대신 `RANK`를 사용합니다.

## ⌨️ 백문이 불여일타 (4)

```
[문제]
1) 고객별 총구매액을 구해 DENSE_RANK로 순위를 매깁니다.
2) 각 고객의 총구매액이 전체 평균 구매액보다 높은지 함께 보여주세요. (CTE + 순위 윈도우 + AVG() OVER ())
```

In [16]:
%%sql
-- [C15] ⌨️ 백문이 불여일타 (4) — DENSE_RANK 순위 + 전체 평균 비교

WITH t AS (
  SELECT
    customer_id,
    SUM(amount) AS total
  FROM project_name.dataset_name.orders
  WHERE amount IS NOT NULL
  GROUP BY customer_id
)
SELECT 
  customer_id, 
  total,
  DENSE_RANK() OVER (ORDER BY total DESC) AS 순위,
  ROUND(AVG(total) OVER(), 2) AS 전체평균,
  CASE WHEN total > AVG(total) OVER () THEN '평균이상' ELSE '평균이하' END AS 평가
FROM t
ORDER BY
  순위,
  customer_id

Running query in 'duckdb'

customer_id,total,순위,전체평균,평가
C013,725000.00,1,271857.14,평균이상
C007,678000.00,2,271857.14,평균이상
C001,380000.00,3,271857.14,평균이상
C012,362000.00,4,271857.14,평균이상
C003,349000.00,5,271857.14,평균이상
C008,314000.00,6,271857.14,평균이상
C002,308000.00,7,271857.14,평균이상
C004,161000.00,8,271857.14,평균이하
C010,142000.00,9,271857.14,평균이하
C005,114000.00,10,271857.14,평균이하


<details>
<summary>(펼치기) 🔑 힌트</summary>

- 1단계 CTE에서 `GROUP BY customer_id`와 `SUM(amount)`으로 고객별 총구매액(`total`)을 만듭니다.
- 순위는 `DENSE_RANK() OVER (ORDER BY total DESC)`, 전체 평균은 `AVG(total) OVER ()` — 둘 다 행을 유지한 채 열로 붙습니다.
- 평균보다 높은지 표시는 `CASE WHEN total > AVG(total) OVER () THEN ... ELSE ... END` 패턴을 씁니다.

</details>

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
WITH t AS (
  SELECT
    customer_id,
    SUM(amount) AS total
  FROM project_name.dataset_name.orders
  WHERE
    amount IS NOT NULL
  GROUP BY
    customer_id
)
SELECT
  customer_id,
  total,
  DENSE_RANK() OVER (ORDER BY total DESC) AS 순위,
  ROUND(AVG(total) OVER (), 0) AS 전체평균,
  CASE WHEN total > AVG(total) OVER () THEN '평균이상' ELSE '평균이하' END AS 평가
FROM t
ORDER BY
  순위 NULLS LAST
```

</details>

## 🚦 짚고 넘어가기

1. `ROW_NUMBER`·`RANK`·`DENSE_RANK`의 동점 처리는 각각 어떻게 다릅니까?
2. `OVER (ORDER BY ...)`의 정렬 기준은 무엇을 정합니까?
3. `PARTITION BY`를 추가하면 순위가 어떻게 바뀝니까?

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

1. `ROW_NUMBER`는 동점이라도 1, 2, 3…으로 유일한 번호를 매기고, `RANK`는 동점에 같은 등수를 준 뒤 다음 등수를 건너뛰며(1, 1, 3), `DENSE_RANK`는 건너뛰지 않습니다(1, 1, 2).
2. 무엇을 기준으로(어느 값, 어느 방향으로) 순위를 매길지를 정합니다. 예를 들어 `ORDER BY revenue DESC`면 매출이 큰 행이 1위가 됩니다.
3. 그룹(파티션)마다 순위가 따로 매겨져, 각 그룹 안에서 1위부터 다시 시작합니다.

</details>

> ⏭️ **다음 학습 예고**  
> 순위 다음으로 많이 쓰는 건 **누적 합계**입니다. "월별 매출의 _누적_"을 윈도우 함수로 구합니다.

# Part 4. 누적·이동 집계 — `SUM() OVER`와 프레임

"이번 달까지의 _누적_ 매출", "최근 3건의 _이동_ 평균" 같은 분석은 _정렬된 순서를 따라가며_ 계산합니다. 윈도우에 `ORDER BY`를 주면 이런 누적 계산이 가능해집니다.

> ❓ **질문하기**  
> 순서를 따라가며 쌓이는 '누적 합계'를 어떻게 구합니까?

## 🧩 개념 잡기

```
월      매출      누적매출 (SUM OVER ORDER BY 월)
2023-09 556,000   556,000
2023-10 765,000   1,321,000   ← 이전까지 + 이번 달
2023-11 786,000   2,107,000
...
```

누적 합계는 계산 순서와 프레임을 함께 적어 의도를 분명하게 만듭니다. `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`는 정렬된 첫 행부터 현재 행까지를 뜻합니다.

```sql
SUM(x) OVER (
  ORDER BY 월
  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
)
```

> ⚠️ **주의하기 — 프레임을 생략하면 동점 행의 범위가 달라질 수 있습니다**  
> DBMS의 기본 프레임은 `ORDER BY` 값이 같은 행을 한꺼번에 포함하는 방식일 수 있습니다. 행 단위 누적을 의도한다면 `ROWS ...`를 명시하고, 정렬 값이 중복될 수 있으면 유일한 두 번째 기준도 추가합니다.

▶️ **코드 실행하기 · 코드 셀 16 [C16]**  
— 월별 매출에 **처음부터 그 달까지의 누적매출**을 함께 붙입니다(프레임 `UNBOUNDED PRECEDING`).

In [17]:
%%sql
-- [C16]
WITH monthly AS (
  SELECT
    date_trunc('month', order_date) /* BigQuery: DATE_TRUNC(order_date, MONTH) */ AS month,
    SUM(amount) AS revenue
  FROM project_name.dataset_name.orders
  WHERE
    amount IS NOT NULL
  GROUP BY
    date_trunc('month', order_date) /* BigQuery: DATE_TRUNC(order_date, MONTH) */
)
SELECT
  month,
  revenue,
  SUM(revenue) OVER (ORDER BY month ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS 누적매출
FROM monthly
ORDER BY
  month NULLS LAST

Running query in 'duckdb'

month,revenue,누적매출
2023-09-01 00:00:00,556000.00,556000.00
2023-10-01 00:00:00,775000.00,1331000.00
2023-11-01 00:00:00,839000.00,2170000.00
2023-12-01 00:00:00,526000.00,2696000.00
2024-01-01 00:00:00,850000.00,3546000.00
2024-02-01 00:00:00,260000.00,3806000.00


> 👀 **결과읽기 — [C16]**  
> `누적매출` 열이 월이 갈수록 계속 쌓입니다. `ORDER BY month`가 계산 순서를, `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`가 첫 달부터 현재 달까지의 계산 범위를 정합니다.

> 📌 **실무 연결하기 — '지금까지 얼마나 쌓였나'**  
> 누적 합계는 _누적 가입자_(서비스), _누적 거래액_(금융), _누적 생산량_(제조)처럼 "지금까지 얼마나 쌓였나"를 보는 모든 KPI에 쓰입니다.

## ⌨️ 백문이 불여일타 (5)

```
[문제] 월별 매출에 '전체 평균 매출'을 나란히 붙여, 각 달이 평균보다 높은지 낮은지 비교할 수 있게 합니다. (AVG(revenue) OVER ())
```

In [18]:
%%sql
-- [C17] ⌨️ 백문이 불여일타 (5) — 월별 매출에 전체 평균 나란히 붙이기

WITH monthly AS (
  SELECT
    CAST(date_trunc('month', order_date) AS DATE) AS month,
    SUM(amount) AS revenue
  FROM project_name.dataset_name.orders
  WHERE
    amount IS NOT NULL
  GROUP BY
    date_trunc('month', order_date)
)
SELECT
  month,
  revenue,
  ROUND(AVG(revenue) OVER(), 2) AS 평균월매출,
  CASE WHEN revenue > AVG(revenue) OVER () THEN '평균이상' ELSE '평균이하' END AS 평가
FROM monthly
ORDER BY
  month NULLS LAST

Running query in 'duckdb'

month,revenue,평균월매출,평가
2023-09-01,556000.00,634333.33,평균이하
2023-10-01,775000.00,634333.33,평균이상
2023-11-01,839000.00,634333.33,평균이상
2023-12-01,526000.00,634333.33,평균이하
2024-01-01,850000.00,634333.33,평균이상
2024-02-01,260000.00,634333.33,평균이하


<details>
<summary>(펼치기) 🔑 힌트</summary>

- 월별 매출 CTE는 본문 누적매출 예제와 같습니다 — `DATE_TRUNC(order_date, MONTH)`로 월을 만들고 `SUM(amount)`을 구합니다.
- 전체 평균은 `AVG(revenue) OVER ()`로 붙입니다. 괄호를 비우면 전체 행이 하나의 창이 되어 모든 행에 같은 평균이 붙습니다.

</details>

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
WITH monthly AS (
  SELECT
    date_trunc('month', order_date) /* BigQuery: DATE_TRUNC(order_date, MONTH) */ AS month,
    SUM(amount) AS revenue
  FROM project_name.dataset_name.orders
  WHERE
    amount IS NOT NULL
  GROUP BY
    date_trunc('month', order_date) /* BigQuery: DATE_TRUNC(order_date, MONTH) */
)
SELECT
  month,
  revenue,
  AVG(revenue) OVER () AS 평균월매출
FROM monthly
ORDER BY
  month NULLS LAST
```

</details>

## 이동 평균 — 프레임으로 '최근 N개' 범위 정하기

누적이 "처음부터 지금까지"였다면, **이동 평균**은 "*최근 몇 개*만"의 평균입니다. `OVER (ORDER BY ... ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)` 처럼 _프레임(범위)_ 을 지정하면, 들쭉날쭉한 값을 부드럽게 다듬어 추세를 봅니다.

▶️ **코드 실행하기 · 코드 셀 18 [C18]**  
— 같은 월별 매출에 **최근 3개월 이동평균**을 붙여, 들쭉날쭉한 값을 다듬어 추세를 봅니다.

In [19]:
%%sql
-- [C18]
WITH monthly AS (
  SELECT
    date_trunc('month', order_date) /* BigQuery: DATE_TRUNC(order_date, MONTH) */ AS month,
    SUM(amount) AS revenue
  FROM project_name.dataset_name.orders
  WHERE
    amount IS NOT NULL
  GROUP BY
    date_trunc('month', order_date) /* BigQuery: DATE_TRUNC(order_date, MONTH) */
)
SELECT
  month,
  revenue,
  ROUND(AVG(revenue) OVER (ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 0) AS 이동평균_3개월
FROM monthly
ORDER BY
  month NULLS LAST

Running query in 'duckdb'

month,revenue,이동평균_3개월
2023-09-01 00:00:00,556000.00,556000.0
2023-10-01 00:00:00,775000.00,665500.0
2023-11-01 00:00:00,839000.00,723333.0
2023-12-01 00:00:00,526000.00,713333.0
2024-01-01 00:00:00,850000.00,738333.0
2024-02-01 00:00:00,260000.00,545333.0


> 👀 **결과읽기 — [C18]**  
> `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW`는 "직전 2행 + 현재 행 = 최근 3개"를 뜻합니다. 월매출이 출렁여도 이동평균은 *완만한 추세선*을 그려줘, 일시적 변동에 휘둘리지 않게 해줍니다.

> 🐼 **pandas와 비교**  
> 이동평균은 pandas의 `df['revenue'].rolling(3).mean()` 과 같은 개념입니다.

## 🚦 짚고 넘어가기

1. `SUM(x) OVER (ORDER BY 월 ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)`은 무엇을 계산합니까?
2. `OVER (ORDER BY ...)` 가 있을 때와 없을 때(`OVER ()`)의 차이는?
3. 누적 합계는 어떤 분석에 유용합니까?

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

1. 월 순서상 첫 행부터 현재 행까지의 누적 합계를 계산합니다.
2. 집계 윈도우 함수에서 `ORDER BY`와 누적 프레임을 지정하면 순서에 따른 누적 값이 되고, `OVER ()`는 전체 집계 값을 모든 행에 똑같이 붙입니다.
3. 월별 목표 달성률 추적, 연간 누적 매출처럼 "지금까지 얼마나 쌓였나"를 보는 분석에 유용합니다.

</details>

> ⏭️ **다음 학습 예고**  
> 누적을 봤으니, 이제 "전월 대비 얼마나 늘었나" 같은 _이웃 행과의 비교_ 를 `LAG`·`LEAD`로 구합니다.

# Part 5. 이전·다음 행 비교 — `LAG` · `LEAD`

"이번 달 매출이 _전월 대비_ 얼마나 늘었나?" 같은 질문은 *바로 앞(또는 뒤) 행의 값*을 가져와야 합니다. `LAG`(앞 행)·`LEAD`(뒤 행)가 그 일을 합니다.

> ❓ **질문하기**  
> 전월 대비 증감처럼, 이웃 행과 비교하려면?

## 🧩 개념 잡기

- `LAG(x)` = **앞(이전) 행의 x 값** 가져오기
- `LEAD(x)` = **뒤(다음) 행의 x 값** 가져오기

```
월      매출      전월매출(LAG)   증감
2023-09 556,000   (없음)          -
2023-10 765,000   556,000         +209,000
2023-11 786,000   765,000         +21,000
```

`LAG(revenue) OVER (ORDER BY month)` — 월 순서대로, 이전 달의 매출을 현재 행에 가져옵니다.

▶️ **코드 실행하기 · 코드 셀 19 [C19]**  
— 월별 매출에 **전월 매출·증감액·증감률(%)** 을 붙입니다 — `LAG`로 이전 행을 가져오고, 분모 0은 `NULLIF`로 방어합니다.

In [20]:
%%sql
-- [C19]
WITH monthly AS (
  SELECT
    date_trunc('month', order_date) /* BigQuery: DATE_TRUNC(order_date, MONTH) */ AS month,
    SUM(amount) AS revenue
  FROM project_name.dataset_name.orders
  WHERE
    amount IS NOT NULL
  GROUP BY
    date_trunc('month', order_date) /* BigQuery: DATE_TRUNC(order_date, MONTH) */
)
SELECT
  month,
  revenue,
  LAG(revenue) OVER (ORDER BY month NULLS LAST) AS 전월매출,
  revenue - LAG(revenue) OVER (ORDER BY month NULLS LAST) AS 증감액,
  ROUND(
    (
      revenue - LAG(revenue) OVER (ORDER BY month NULLS LAST)
    ) * 100.0 / NULLIF(NULLIF(LAG(revenue) OVER (ORDER BY month NULLS LAST), 0), 0),
    1
  ) AS 증감률_pct
FROM monthly
ORDER BY
  month NULLS LAST

Running query in 'duckdb'

month,revenue,전월매출,증감액,증감률_pct
2023-09-01 00:00:00,556000.00,None,None,None
2023-10-01 00:00:00,775000.00,556000.00,219000.00,39.4
2023-11-01 00:00:00,839000.00,775000.00,64000.00,8.3
2023-12-01 00:00:00,526000.00,839000.00,-313000.00,-37.3
2024-01-01 00:00:00,850000.00,526000.00,324000.00,61.6
2024-02-01 00:00:00,260000.00,850000.00,-590000.00,-69.4


> 👀 **결과읽기 — [C19]**  
> `전월매출`은 `LAG`로 한 칸 위의 매출을 가져온 것이고, `증감액`·`증감률_pct`는 그걸로 계산했습니다. 첫 달은 이전이 없어 `NULL`입니다. `NULLIF(..., 0)`은 0으로 나누는 오류를 막는 표준 기법입니다.

> 🐼 **pandas와 비교**  
> `LAG(revenue)` ↔ `df['revenue'].shift(1)`. 전월 대비 증감률은 `df['revenue'].pct_change()`와 같은 개념입니다.

> 📌 **실무 연결하기 — 시계열 비교**  
> `LAG`/`LEAD`는 _전일 대비 주가 변동_(금융), _전월 대비 트래픽_(서비스), _직전 측정 대비 변화_(IoT/제조)처럼 시계열 비교의 핵심입니다.

## ⌨️ 백문이 불여일타 (6)

```
[문제] 위 월별 매출에서 LAG 대신 LEAD를 사용해, 각 달에 '다음 달 매출'을 나란히 붙입니다. (마지막 달은 NULL이 됩니다.)
```

In [21]:
%%sql
-- [C20] ⌨️ 백문이 불여일타 (6) — LEAD로 다음 달 매출 붙이기

WITH monthly AS (
  SELECT
    CAST(date_trunc('month', order_date) AS DATE) AS month,
    SUM(amount) AS revenue
  FROM project_name.dataset_name.orders
  WHERE
    amount IS NOT NULL
  GROUP BY
    date_trunc('month', order_date)
)
SELECT
  month,
  revenue,
  LEAD(revenue) OVER (ORDER BY month NULLS LAST) AS 다음달매출,
  LEAD(revenue) OVER (ORDER BY month NULLS LAST) - revenue AS 증감액,
  ROUND(
    (
      LEAD(revenue) OVER (ORDER BY month NULLS LAST) - revenue
    ) * 100.0 / NULLIF(revenue, 0),
    1
  ) AS 증감률_pct
FROM monthly
ORDER BY
  month NULLS LAST

Running query in 'duckdb'

month,revenue,다음달매출,증감액,증감률_pct
2023-09-01,556000.00,775000.00,219000.00,39.4
2023-10-01,775000.00,839000.00,64000.00,8.3
2023-11-01,839000.00,526000.00,-313000.00,-37.3
2023-12-01,526000.00,850000.00,324000.00,61.6
2024-01-01,850000.00,260000.00,-590000.00,-69.4
2024-02-01,260000.00,None,None,None


<details>
<summary>(펼치기) 🔑 힌트</summary>

- 본문의 `LAG` 예제에서 함수 이름만 `LEAD`로 바꾸면 됩니다.
- `LEAD(revenue) OVER (ORDER BY month)`는 정렬 순서상 다음 행의 값을 가져옵니다. 마지막 달은 다음 행이 없어 `NULL`이 됩니다.

</details>

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
WITH monthly AS (
  SELECT
    date_trunc('month', order_date) /* BigQuery: DATE_TRUNC(order_date, MONTH) */ AS month,
    SUM(amount) AS revenue
  FROM project_name.dataset_name.orders
  WHERE
    amount IS NOT NULL
  GROUP BY
    date_trunc('month', order_date) /* BigQuery: DATE_TRUNC(order_date, MONTH) */
)
SELECT
  month,
  revenue,
  LEAD(revenue) OVER (ORDER BY month NULLS LAST) AS 다음달매출
FROM monthly
ORDER BY
  month NULLS LAST
```

</details>

## 🚦 짚고 넘어가기

다음 질문에 답할 수 있으면 다음 Part로 넘어가세요. 틀려도 괜찮습니다.

1. `LAG`와 `LEAD`는 각각 어느 방향의 행 값을 가져옵니까?
2. 정렬한 첫 번째 행의 `LAG` 결과는 왜 비어 있습니까?
3. 전월 대비 증감률을 구할 때 `NULLIF`를 씌우는 이유는 무엇입니까?

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

1. `LAG`는 **앞(이전) 행**, `LEAD`는 **뒤(다음) 행**의 값을 가져옵니다. "전월 대비"처럼 과거와 비교할 땐 `LAG`입니다.
2. 첫 행에는 **이전 행이 없기 때문**입니다. 그래서 `NULL`이 나오고, 이를 이용한 계산 결과도 `NULL`이 됩니다. 오류가 아니라 정상 동작입니다.
3. **0으로 나누는 오류를 막기 위해서**입니다. 전월 매출이 0이면 `NULLIF(전월매출, 0)`이 `NULL`을 돌려주고, 어떤 값을 `NULL`로 나누면 결과도 `NULL`이 되어 쿼리가 멈추지 않습니다.

</details>

> ⏭️ **다음 학습 예고**  
> 순위·누적·전후 비교까지 익혔습니다. 다음 Part에서는 창의 *첫 행·마지막 행 값을 참조*하고(`FIRST_VALUE`·`LAST_VALUE`), 정렬한 순서를 _몇 개 구간으로 나누는_(`NTILE`) 법을 배웁니다. 이어지는 Part 7에서는 `GROUP BY`를 확장해 **소계·총계 행을 자동으로 만드는** `ROLLUP`·`CUBE`를 다룹니다.

# Part 6. 값 참조와 구간 나누기 — `FIRST_VALUE` · `LAST_VALUE` · `NTILE`

순위·누적·전후 비교까지 배웠습니다. 윈도우 함수에는 실무에서 자주 쓰이는 것이 셋 더 있습니다. **기준값과 비교하기**(`FIRST_VALUE`·`LAST_VALUE`)와 **구간으로 나누기**(`NTILE`)입니다.

> ❓ **질문하기**  
> "1위 대비 몇 %인가", "상위 20%인가"를 SQL로 어떻게 구합니까?

## 🧩 개념 잡기

- `FIRST_VALUE(x)` = 정렬한 창에서 **첫 번째 행의 x**
- `LAST_VALUE(x)` = 정렬한 창에서 **마지막 행의 x**
- `NTILE(n)` = 정렬한 순서대로 **n개 구간에 균등 배분**한 번호(1~n)

```
카테고리별 매출 내림차순
 상품      매출     FIRST_VALUE(매출)   1위대비
 A       500,000      500,000          100%
 B       300,000      500,000           60%
 C       100,000      500,000           20%
                        ▲ 창 전체에서 '첫 행'의 값이 모든 행에 붙습니다
```

▶️ **코드 실행하기 · 코드 셀 21 [C21]**  
— 카테고리별 매출에 **1위 카테고리의 매출과 그 대비 비율(%)** 을 붙입니다(`FIRST_VALUE`).

In [22]:
%%sql
-- [C21]
WITH cat AS (
  SELECT
    p.category,
    SUM(oi.quantity * oi.unit_price) AS revenue
  FROM project_name.dataset_name.order_items AS oi
  JOIN project_name.dataset_name.products AS p
    ON oi.product_id = p.product_id
  GROUP BY
    p.category
)
SELECT
  category,
  revenue,
  FIRST_VALUE(revenue) OVER (ORDER BY revenue DESC) AS 최고매출,
  ROUND(
    revenue * 100.0 / FIRST_VALUE(revenue) OVER (ORDER BY revenue DESC),
    1
  ) AS 최고대비_pct
FROM cat
ORDER BY
  revenue DESC

Running query in 'duckdb'

category,revenue,최고매출,최고대비_pct
Sandals,1081000.00,1081000.00,100.0
Sneakers,989000.00,1081000.00,91.5
Boots,635000.00,1081000.00,58.7
Running,600000.00,1081000.00,55.5
Accessory,501000.00,1081000.00,46.3


> 👀 **결과읽기 — [C21]**  
> `FIRST_VALUE(revenue) OVER (ORDER BY revenue DESC)`는 "매출 내림차순으로 줄 세웠을 때 **맨 앞 값**"입니다. 그 값이 모든 행에 똑같이 붙으므로 *1위 대비 비율*을 바로 계산할 수 있습니다.

> ⚠️ **주의하기 — `LAST_VALUE`는 함정이 있습니다**  
> 윈도우의 기본 프레임이 "처음부터 현재 행까지"라서, `LAST_VALUE`를 그냥 쓰면 *창 전체의 마지막*이 아니라 *현재 행*이 나옵니다. 전체 마지막을 원하면 프레임을 명시해야 합니다.
>
> ```sql
> LAST_VALUE(revenue) OVER (
>   ORDER BY revenue DESC
>   ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
> )
> ```

## `NTILE` — 상위 몇 %인지 등급으로 나누기

`NTILE(5)`는 정렬한 순서대로 **다섯 덩어리**로 나눠 1~5번을 매깁니다. 1번이 가장 앞(상위 20%)입니다. 다음 시간에 배울 **RFM 등급**이 바로 이 함수로 만들어집니다.

▶️ **코드 실행하기 · 코드 셀 22 [C22]**  
— 고객별 총 구매액을 **구매력 4분위(`NTILE(4)`)와 순위**로 함께 나눕니다.

In [23]:
%%sql
-- [C22]
WITH cust AS (
  SELECT
    customer_id,
    SUM(amount) AS total_spent
  FROM project_name.dataset_name.orders
  WHERE
    amount IS NOT NULL
  GROUP BY
    customer_id
)
SELECT
  customer_id,
  total_spent,
  NTILE(4) OVER (ORDER BY total_spent DESC) AS 구매력_4분위,
  RANK() OVER (ORDER BY total_spent DESC) AS 순위
FROM cust
ORDER BY
  total_spent DESC

Running query in 'duckdb'

customer_id,total_spent,구매력_4분위,순위
C013,725000.00,1,1
C007,678000.00,1,2
C001,380000.00,1,3
C012,362000.00,1,4
C003,349000.00,2,5
C008,314000.00,2,6
C002,308000.00,2,7
C004,161000.00,2,8
C010,142000.00,3,9
C005,114000.00,3,10


> 👀 **결과읽기 — [C22]**  
> `구매력_4분위`가 **1이면 상위 25%**, 4면 하위 25%입니다. `RANK`가 *개별 등수*라면 `NTILE`은 *등급 구간*입니다. 고객을 몇 개 그룹으로 나눠 다르게 대응할 때 훨씬 쓸모 있습니다.

> 💡 **핵심짚기 — `NTILE`은 인원을 최대한 균등하게 나눕니다**  
> 10명을 `NTILE(4)`로 나누면 3·3·2·2명이 됩니다. *값의 구간*이 아니라 *인원수*를 기준으로 자른다는 점을 기억합니다.

> 📌 **실무 연결하기 — 줄 세운 뒤 등급으로 자르기**  
> 성적 등급(교육), 신용 등급 구간(금융), 위험도 분위(보험) — "줄 세운 뒤 몇 등급으로 자르기"는 어디서나 씁니다.

## 🚦 짚고 넘어가기

1. `FIRST_VALUE`로 "1위 대비 비율"을 구할 수 있는 이유는?
2. `LAST_VALUE`를 그냥 쓰면 왜 기대와 다른 값이 나옵니까?
3. `RANK`와 `NTILE`의 차이는 무엇입니까?

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

1. 창 전체의 첫 행 값이 **모든 행에 동일하게** 붙기 때문입니다. 각 행의 값을 그 공통 기준값으로 나누면 비율이 됩니다.
2. 기본 프레임이 *처음부터 현재 행까지*라서 "마지막 행"이 곧 **현재 행**이 되기 때문입니다. `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING`으로 프레임을 열어 줘야 합니다.
3. `RANK`는 **개별 등수**(1, 2, 3…)를, `NTILE(n)`은 **n개 등급 구간**(1~n)을 줍니다. 100명이면 `RANK`는 1~100, `NTILE(4)`는 1~4입니다.

</details>

> ⏭️ **다음 학습 예고**  
> 윈도우 함수는 행을 유지한 채 계산했습니다. 이번엔 반대로 `GROUP BY`를 확장해 **소계·총계 행을 자동으로 만드는** 법을 배웁니다.

# Part 7. 소계·총계 — `ROLLUP` · `CUBE` · `GROUPING SETS`

리포트를 만들다 보면 _"카테고리별 매출 + 맨 아래 전체 합계"_ 같은 표가 필요합니다. 지금까지 배운 방법이라면 집계 쿼리를 두 번 짜서 `UNION ALL`로 붙여야 했습니다.

`ROLLUP`은 그 일을 **한 줄로** 해 줍니다.

> ❓ **질문하기**  
> 소계와 총계 행을 SQL이 알아서 만들게 하려면?

## 🧩 개념 잡기

`GROUP BY ROLLUP(a, b)`는 **묶는 기준을 하나씩 걷어내며** 집계를 반복합니다.

```
GROUP BY ROLLUP(category, status)
  ①  category + status 별  ← 가장 잘게
  ②  category 별            ← status를 걷어낸 소계
  ③  (전체)                 ← 다 걷어낸 총계
```

- `ROLLUP(a, b)` — **계층 소계**: (a,b) → (a) → 전체
- `CUBE(a, b)` — **모든 조합**: (a,b) → (a) → (b) → 전체
- `GROUPING SETS((a),(b))` — **원하는 조합만** 골라서

소계 행에서는 걷어낸 열이 `NULL`로 채워집니다.

▶️ **코드 실행하기 · 코드 셀 23 [C23]**  
— 카테고리 × 상태 매출을 **`ROLLUP`으로 묶어**, 조합별 매출과 **카테고리 소계·전체 총계까지 한 번에** 구합니다.

In [24]:
%%sql
-- [C23]
SELECT
  p.category,
  o.status,
  SUM(oi.quantity * oi.unit_price) AS 매출
FROM project_name.dataset_name.order_items AS oi
JOIN project_name.dataset_name.products AS p
  ON oi.product_id = p.product_id
JOIN project_name.dataset_name.orders AS o
  ON oi.order_id = o.order_id
GROUP BY
  ROLLUP(p.category, o.status)
ORDER BY
  p.category NULLS LAST,
  o.status NULLS LAST

Running query in 'duckdb'

category,status,매출
Accessory,Paid,286000.00
Accessory,Shipped,215000.00
Accessory,None,501000.00
Boots,Paid,466000.00
Boots,Placed,32000.00
Boots,Returned,45000.00
Boots,Shipped,92000.00
Boots,None,635000.00
Running,Paid,320000.00
Running,Placed,38000.00


> 👀 **결과읽기 — [C23]**  
> 결과를 보면 세 종류의 행이 섞여 있습니다. `category`와 `status`가 **둘 다 있는 행**(가장 잘게 쪼갠 집계), `status`만 `NULL`인 행(카테고리 소계), **둘 다 `NULL`인 행**(전체 총계)입니다.

> ⚠️ **주의하기 — 원본 `NULL`과 소계의 `NULL`은 `GROUPING`으로 구분합니다**  
> `category IS NULL`만 확인하면 원래 카테고리가 비어 있던 행도 총계로 잘못 표시할 수 있습니다. `GROUPING(category)`는 해당 열이 소계·총계를 만들기 위해 걷어졌으면 1, 실제 그룹 값이면 0을 반환합니다.

▶️ **코드 실행하기 · 코드 셀 24 [C24]**  
— 소계 행의 `NULL`과 **원래 값이 비어 있던 `NULL`을 `GROUPING`으로 구분해** 표시합니다.

In [25]:
%%sql
-- [C24]
SELECT
  CASE
    WHEN GROUPING(p.category) = 1 THEN '── 전체 합계'
    WHEN p.category IS NULL THEN '(카테고리 미입력)'
    ELSE p.category
  END AS 구분,
  SUM(oi.quantity * oi.unit_price) AS 매출,
  COUNT(*) AS 항목수
FROM project_name.dataset_name.order_items AS oi
JOIN project_name.dataset_name.products AS p
  ON oi.product_id = p.product_id
GROUP BY
  ROLLUP(p.category)
ORDER BY
  GROUPING(p.category),
  p.category NULLS LAST

Running query in 'duckdb'

구분,매출,항목수
Accessory,501000.00,3
Boots,635000.00,6
Running,600000.00,6
Sandals,1081000.00,6
Sneakers,989000.00,6
── 전체 합계,3806000.00,27


> 👀 **결과읽기 — [C24]**  
> `GROUPING(p.category) = 1`인 행만 `── 전체 합계`로 표시됩니다. 원본 카테고리가 실제로 `NULL`인 행은 `(카테고리 미입력)`으로 따로 구분됩니다. **엑셀 피벗테이블의 '총합계' 행을 SQL로 만든 것**과 같습니다.

> 💡 **핵심짚기 — `CUBE`는 언제 쓰나요**  
> 카테고리별 소계와 **상태별 소계를 동시에** 보고 싶을 때입니다. `ROLLUP`은 왼쪽 기준의 계층 소계만 만들지만, `CUBE`는 모든 조합의 소계를 만듭니다. 열이 늘수록 결과가 급격히 커지니 필요한 조합만 `GROUPING SETS`로 지정하는 편이 실무적입니다.

> 📌 **실무 연결하기 — 경영 리포트의 소계 행**  
> 부서별·직급별 인건비 소계(HR), 지점별·상품별 판매 소계(리테일), 진료과별·연령대별 환자 수(헬스케어) — 경영 리포트는 거의 항상 소계 행을 요구합니다.

## 🚦 짚고 넘어가기

1. `ROLLUP(a, b)`가 만드는 집계 수준 세 가지를 말합니다.
2. 소계 행에서 걷어낸 열은 어떤 값으로 채워집니까?
3. `ROLLUP`과 `CUBE`의 차이는?

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

1. **(a, b) 조합별 → (a)별 소계 → 전체 총계** 순으로, 오른쪽 기준부터 하나씩 걷어내며 집계합니다.
2. **`NULL`** 로 채워집니다. 원본 `NULL`과 구분하려면 `GROUPING(열)`을 사용하고, 그 결과가 1인 행에만 소계·총계 라벨을 붙입니다.
3. `ROLLUP`은 **계층 소계**만((a,b), (a), 전체), `CUBE`는 **모든 조합**((a,b), (a), (b), 전체)을 만듭니다.

</details>

> ⏭️ **다음 학습 예고**  
> CTE와 윈도우 함수를 5문항으로 정리합니다.

# 📒 연습 문제 — CTE와 윈도우 함수 (5문항)

오늘 배운 것을 모아 훈련합니다. [1]→[2]→[3]은 난이도가 이어지니 순서대로 푸세요.

**풀이 방법:** 문제를 먼저 풀고 실행한 뒤 바로 아래의 예시 답안을 펼쳐 비교합니다. 막히면 앞 Part의 예제 셀을 다시 확인합니다.

In [26]:
%%sql
-- [C25] 📒 연습 문제 — CTE와 윈도우 함수 (1/5)

-- 문제: CTE로 고객별 총구매액을 구한 뒤, 30만원 이상인 고객만 조회합니다.

WITH c AS (
    SELECT customer_id, SUM(amount) AS 총구매액
    FROM project_name.dataset_name.orders 
    WHERE amount IS NOT NULL
    GROUP BY customer_id 
)
SELECT * FROM c
WHERE 총구매액 >= 300000
ORDER BY 총구매액 DESC

Running query in 'duckdb'

customer_id,총구매액
C013,725000.00
C007,678000.00
C001,380000.00
C012,362000.00
C003,349000.00
C008,314000.00
C002,308000.00


<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
-- [1] CTE로 단계 나누기
WITH cust AS (
  SELECT customer_id, SUM(amount) AS total_spent
  FROM project_name.dataset_name.orders
  WHERE amount IS NOT NULL
  GROUP BY customer_id
)
SELECT * FROM cust
WHERE total_spent >= 300000
ORDER BY total_spent DESC;
```

</details>

In [27]:
%%sql
-- [C26] 📒 연습 문제 — CTE와 윈도우 함수 (2/5)

-- 문제: 각 카테고리 안에서 매출 1위 상품만 뽑으세요. (PARTITION BY + ROW_NUMBER)

WITH prod_rev AS (
    SELECT
        p.category,
        p.product_name,
        SUM(oi.quantity * oi.unit_price) AS revenue
    FROM project_name.dataset_name.order_items AS oi
    JOIN project_name.dataset_name.products AS p
      ON oi.product_id = p.product_id
    GROUP BY p.category, p.product_name 
), ranked AS (
    SELECT category, product_name, revenue, 
           ROW_NUMBER() OVER (PARTITION BY category ORDER BY revenue DESC) AS rank
    FROM prod_rev 
)
SELECT category, product_name, revenue FROM ranked
WHERE rank = 1
ORDER BY revenue DESC


Running query in 'duckdb'

category,product_name,revenue
Sneakers,캔버스 스니커즈,624000.00
Sandals,여름 샌들,551000.00
Accessory,양말 세트,501000.00
Boots,워커 부츠,410000.00
Running,트레일 러너,361000.00


<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
-- [2] 그룹별 1등만 — 순위를 매긴 뒤 바깥에서 거릅니다
WITH prod AS (
  SELECT p.category, p.product_name, SUM(oi.quantity * oi.unit_price) AS revenue
  FROM project_name.dataset_name.order_items AS oi
  JOIN project_name.dataset_name.products AS p ON oi.product_id = p.product_id
  GROUP BY p.category, p.product_name
),
ranked AS (
  SELECT category, product_name, revenue,
         ROW_NUMBER() OVER (PARTITION BY category ORDER BY revenue DESC) AS rn
  FROM prod
)
SELECT category, product_name, revenue
FROM ranked
WHERE rn = 1
ORDER BY revenue DESC;
```

</details>

In [28]:
%%sql
-- [C27] 📒 연습 문제 — CTE와 윈도우 함수 (3/5)

-- 문제: 월별 매출과 누적 매출을 함께 조회합니다.

WITH monthly AS (
    SELECT CAST(date_trunc('month', order_date) AS DATE) AS month, SUM(amount) AS revenue
    FROM project_name.dataset_name.orders 
    WHERE amount IS NOT NULL
    GROUP BY month
)
SELECT month, revenue, SUM(revenue) OVER (ORDER BY month) AS 누적매출
FROM monthly
ORDER BY month

Running query in 'duckdb'

month,revenue,누적매출
2023-09-01,556000.00,556000.00
2023-10-01,775000.00,1331000.00
2023-11-01,839000.00,2170000.00
2023-12-01,526000.00,2696000.00
2024-01-01,850000.00,3546000.00
2024-02-01,260000.00,3806000.00


<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
-- [3] 누적 합계
WITH monthly AS (
  SELECT date_trunc('month', order_date) /* BigQuery: DATE_TRUNC(order_date, MONTH) */ AS month,
         SUM(amount) AS revenue
  FROM project_name.dataset_name.orders
  WHERE amount IS NOT NULL
  GROUP BY 1
)
SELECT month, revenue,
       SUM(revenue) OVER (ORDER BY month) AS 누적매출
FROM monthly
ORDER BY month;
```

**확인할 점**

- **`GROUP BY 1`** — `SELECT`의 첫 번째 열로 묶으라는 뜻입니다. 긴 식을 두 번 쓰지 않아도 되어 편하지만, 열 순서를 바꾸면 조용히 깨지니 주의합니다.

</details>

In [29]:
%%sql
-- [C28] 📒 연습 문제 — CTE와 윈도우 함수 (4/5)

-- 문제: 월별 매출에 전월 매출과 증감률(%)을 붙이세요. (LAG)

WITH monthly AS (
    SELECT CAST(date_trunc('month', order_date) AS DATE) AS month, SUM(amount) AS revenue
    FROM project_name.dataset_name.orders 
    WHERE amount IS NOT NULL
    GROUP BY month
)
SELECT month, revenue AS 월별매출, LAG(revenue) OVER (ORDER BY month) AS 전월매출,
       ROUND((revenue - LAG(revenue) OVER (ORDER BY month)) * 100 
            / NULLIF (LAG(revenue) OVER (ORDER BY month), 0), 1) AS 증감률_pct
FROM monthly
ORDER BY month

Running query in 'duckdb'

month,월별매출,전월매출,증감률_pct
2023-09-01,556000.00,None,None
2023-10-01,775000.00,556000.00,39.4
2023-11-01,839000.00,775000.00,8.3
2023-12-01,526000.00,839000.00,-37.3
2024-01-01,850000.00,526000.00,61.6
2024-02-01,260000.00,850000.00,-69.4


<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
-- [4] 전월 대비
WITH monthly AS (
  SELECT date_trunc('month', order_date) /* BigQuery: DATE_TRUNC(order_date, MONTH) */ AS month,
         SUM(amount) AS revenue
  FROM project_name.dataset_name.orders
  WHERE amount IS NOT NULL
  GROUP BY 1
)
SELECT month, revenue,
       LAG(revenue) OVER (ORDER BY month) AS 전월매출,
       ROUND((revenue - LAG(revenue) OVER (ORDER BY month)) * 100.0
             / NULLIF(LAG(revenue) OVER (ORDER BY month), 0), 1) AS 증감률_pct
FROM monthly
ORDER BY month;
```

**확인할 점**

- **0으로 나누기** — `NULLIF(전월매출, 0)`이 없으면 전월 매출이 0일 때 오류가 납니다. 나눗셈에는 습관적으로 `NULLIF`를 씌웁니다.

</details>

In [30]:
%%sql
-- [C29] 📒 연습 문제 — CTE와 윈도우 함수 (5/5)

-- 문제: 고객별 총구매액을 NTILE(3)으로 상·중·하 등급으로 나누세요.

WITH c AS (
    SELECT customer_id, SUM(amount) AS 총구매액
    FROM project_name.dataset_name.orders 
    WHERE amount IS NOT NULL
    GROUP BY customer_id 
)
SELECT customer_id, 총구매액,
       CASE NTILE(3) OVER (ORDER BY 총구매액 DESC) 
        WHEN 1 THEN '상' WHEN 2 THEN '중' ELSE '하'
       END AS 등급 
FROM c
ORDER BY 총구매액 DESC


Running query in 'duckdb'

customer_id,총구매액,등급
C013,725000.00,상
C007,678000.00,상
C001,380000.00,상
C012,362000.00,상
C003,349000.00,상
C008,314000.00,중
C002,308000.00,중
C004,161000.00,중
C010,142000.00,중
C005,114000.00,중


<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
-- [5] 3등급으로 나누기
WITH cust AS (
  SELECT customer_id, SUM(amount) AS total_spent
  FROM project_name.dataset_name.orders
  WHERE amount IS NOT NULL
  GROUP BY customer_id
)
SELECT customer_id, total_spent,
       NTILE(3) OVER (ORDER BY total_spent DESC) AS 등급
FROM cust
ORDER BY total_spent DESC;
```

</details>

# 🧪 종합 실습 — 월별 누적 매출과 전월 대비 증감률

오늘 배운 CTE와 윈도우 함수를 모아, 모두몰의 **"성장 추이 분석 노트"** 를 완성합니다. 한 번에 _월 누적매출 + 전월대비 증감률 + (참고) 순위_ 까지 담습니다.

**질문 1.** 월별 매출, 누적매출, 전월 대비 증감률을 한 표에 담으세요.

In [31]:
%%sql
-- [C30] 🧪 종합 실습 — 월별 누적 매출과 전월 대비 증감률 (1/2)

WITH monthly AS (
    SELECT CAST(date_trunc('month', order_date) AS DATE) AS month, SUM(amount) AS revenue
    FROM project_name.dataset_name.orders 
    WHERE amount IS NOT NULL
    GROUP BY month
)
SELECT month, revenue AS 월별매출, 
       SUM(revenue) OVER (ORDER BY month) AS 누적매출,
       LAG(revenue) OVER (ORDER BY month) AS 전월매출,
       ROUND((revenue - LAG(revenue) OVER (ORDER BY month)) * 100 
            / NULLIF (LAG(revenue) OVER (ORDER BY month), 0), 1) AS 증감률_pct
FROM monthly
ORDER BY month

Running query in 'duckdb'

month,월별매출,누적매출,전월매출,증감률_pct
2023-09-01,556000.00,556000.00,None,None
2023-10-01,775000.00,1331000.00,556000.00,39.4
2023-11-01,839000.00,2170000.00,775000.00,8.3
2023-12-01,526000.00,2696000.00,839000.00,-37.3
2024-01-01,850000.00,3546000.00,526000.00,61.6
2024-02-01,260000.00,3806000.00,850000.00,-69.4


<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
WITH monthly AS (
  SELECT
    date_trunc('month', order_date) /* BigQuery: DATE_TRUNC(order_date, MONTH) */ AS month,
    SUM(amount) AS revenue
  FROM project_name.dataset_name.orders
  WHERE
    amount IS NOT NULL
  GROUP BY
    date_trunc('month', order_date) /* BigQuery: DATE_TRUNC(order_date, MONTH) */
)
SELECT
  month,
  revenue,
  SUM(revenue) OVER (ORDER BY month) AS 누적매출,
  LAG(revenue) OVER (ORDER BY month NULLS LAST) AS 전월매출,
  ROUND(
    (
      revenue - LAG(revenue) OVER (ORDER BY month NULLS LAST)
    ) * 100.0 / NULLIF(NULLIF(LAG(revenue) OVER (ORDER BY month NULLS LAST), 0), 0),
    1
  ) AS 증감률_pct
FROM monthly
ORDER BY
  month NULLS LAST
```

</details>

**질문 2.** 월별 매출과 누적매출을 한 그래프에 그려, 성장 추이를 시각화합니다.

In [32]:
%%sql
-- [C31] 🧪 종합 실습 — 월별 누적 매출과 전월 대비 증감률 (2/2)

WITH monthly AS (
    SELECT CAST(date_trunc('month', order_date) AS DATE) AS month, SUM(amount) AS revenue
    FROM project_name.dataset_name.orders 
    WHERE amount IS NOT NULL
    GROUP BY month
)
SELECT month, revenue AS 월별매출, 
       SUM(revenue) OVER (ORDER BY month) AS 누적매출,
FROM monthly
ORDER BY month

Running query in 'duckdb'

month,월별매출,누적매출
2023-09-01,556000.00,556000.00
2023-10-01,775000.00,1331000.00
2023-11-01,839000.00,2170000.00
2023-12-01,526000.00,2696000.00
2024-01-01,850000.00,3546000.00
2024-02-01,260000.00,3806000.00


<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
WITH monthly AS (
  SELECT
    date_trunc('month', order_date) /* BigQuery: DATE_TRUNC(order_date, MONTH) */ AS month,
    SUM(amount) AS revenue
  FROM project_name.dataset_name.orders
  WHERE
    amount IS NOT NULL
  GROUP BY
    date_trunc('month', order_date) /* BigQuery: DATE_TRUNC(order_date, MONTH) */
)
SELECT
  month,
  revenue,
  SUM(revenue) OVER (ORDER BY month) AS cum_revenue
FROM monthly
ORDER BY
  month NULLS LAST
```

</details>

> 📊 이 결과는 LMS에서는 표로 확인하고, BigQuery에서는 콘솔 차트(또는 Looker Studio)로 시각화할 수 있습니다.

> 👀 **결과읽기 — [C31]**  
> 막대(월매출)는 오르내려도, 선(누적매출)은 계속 우상향합니다. 윈도우 함수 하나로 "흐름"과 "쌓임"을 동시에 보여준 겁니다.

# 🎯 오늘의 퀴즈

배운 내용을 잠깐 확인해보겠습니다. 틀려도 괜찮습니다!

## 개념 퀴즈

1. `GROUP BY`와 윈도우 함수의 가장 큰 차이는 _결과 행 수_ 측면에서 무엇입니까?
2. `RANK()`와 `DENSE_RANK()`는 동점 다음 등수를 각각 어떻게 처리합니까?
3. "전월 대비"를 구할 때 `LAG`와 `LEAD` 중 무엇을 씁니까?
4. CTE(`WITH`)를 쓰면 서브쿼리에 비해 어떤 점이 좋아집니까?

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

1. `GROUP BY`는 행을 **줄이고(접고)**, 윈도우 함수는 행을 **유지**합니다.
2. `RANK`는 다음 등수를 **건너뛰고**(1,1,3), `DENSE_RANK`는 **안 건너뜁니다**(1,1,2).
3. **`LAG`** (이전=앞 행의 값).
4. 깊은 괄호 대신 **이름 붙인 단계**로 위에서 아래로 읽혀 가독성이 좋아집니다.
</details>

## 코드 퀴즈

**문제:** `order_items`+`products`로 카테고리별 매출을 구한 뒤, `RANK()`로 **매출 순위**를 매겨 보여주세요. (CTE + 윈도우)

In [33]:
%%sql
-- [C32] 코드 퀴즈 — 카테고리별 매출 순위 매기기


WITH cat_rev AS (
    SELECT
        p.category,
        SUM(oi.quantity * oi.unit_price) AS revenue
    FROM project_name.dataset_name.order_items AS oi
    JOIN project_name.dataset_name.products AS p
      ON oi.product_id = p.product_id
    GROUP BY p.category 
)
SELECT category, revenue, RANK() OVER (ORDER BY revenue DESC) AS 순위 FROM cat_rev
ORDER BY 순위

Running query in 'duckdb'

category,revenue,순위
Sandals,1081000.00,1
Sneakers,989000.00,2
Boots,635000.00,3
Running,600000.00,4
Accessory,501000.00,5


<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
WITH cat AS (
  SELECT
    p.category,
    SUM(oi.quantity * oi.unit_price) AS revenue
  FROM project_name.dataset_name.order_items AS oi
  JOIN project_name.dataset_name.products AS p
    ON oi.product_id = p.product_id
  GROUP BY
    p.category
)
SELECT
  category,
  revenue,
  RANK() OVER (ORDER BY revenue DESC) AS 순위
FROM cat
ORDER BY
  순위 NULLS LAST
```

</details>

# 🎓 오늘의 정리

## 오늘 배운 것이 어떻게 이어지는가

```
[Part 1] WITH(CTE)        (단계를 이름으로 — 가독성)
   ↓
[Part 2] OVER             (행을 유지하며 그룹 계산)
   ↓
[Part 3] 순위 함수          (ROW_NUMBER / RANK / DENSE_RANK)
   ↓
[Part 4] SUM OVER         (순서 따라 누적·이동)
   ↓
[Part 5] LAG / LEAD       (전·후 행 비교 — 전월 대비)
   ↓
[다음 시간] 코호트·퍼널·RFM    (비즈니스 분석 — 오늘 도구를 총동원)
```

## 한 장 정리 — 오늘의 쿼리 패턴 치트시트

| 하고 싶은 것        | SQL                                              |
| ------------------- | ------------------------------------------------ |
| 단계에 이름 붙이기  | `WITH 이름 AS (SELECT ...) SELECT ... FROM 이름` |
| 행 유지하며 그룹 합 | `SUM(x) OVER (PARTITION BY g)`                   |
| 등수 매기기         | `RANK() OVER (ORDER BY x DESC)`                  |
| 그룹별 등수         | `RANK() OVER (PARTITION BY g ORDER BY x DESC)`   |
| 누적 합계           | `SUM(x) OVER (ORDER BY 순서열)`                  |
| 이전/다음 행 값     | `LAG(x) OVER (ORDER BY 순서열)` / `LEAD(x)`      |
| 0 나누기 방지       | `a / NULLIF(b, 0)`                               |

> 💡 **핵심짚기 — 윈도우 함수의 구조**  
> 윈도우 함수 = `함수() OVER (PARTITION BY ... ORDER BY ...)`. `PARTITION BY`는 그룹, `ORDER BY`는 순서를 정합니다.

## 📝 오늘의 과제

**성장 추이 분석 노트** — 모두몰 데이터로 _시간/순위_ 분석 질문 5개를 만들고 SQL로 답합니다.

**제출물:**

1. CTE·윈도우 함수가 필요한 질문 5개 (예: "월 누적매출 추이는?", "카테고리별 베스트셀러 순위는?")
2. 각 질문에 답하는 SQL
3. 결과에서 발견한 인사이트 1줄씩

**조건:**

- `WITH`(CTE)를 최소 2개 질문에 사용
- 순위 함수(`RANK`/`ROW_NUMBER`/`DENSE_RANK`)와 `LAG`/`LEAD`를 각각 최소 1번 사용
- `SUM() OVER (...)` 누적을 최소 1번 사용

**평가 기준**

| 항목     | 기준                              |
| -------- | --------------------------------- |
| 정확성   | 윈도우·CTE가 의도대로 동작하는가  |
| 가독성   | CTE로 단계를 읽기 좋게 구성했는가 |
| 인사이트 | 추이·순위에서 시사점을 읽어냈는가 |

**제출:** 개인 GitHub 저장소 main에 커밋·푸시하고 링크를 제출합니다.

In [34]:
%%sql
-- [Q1] 월별 매출이 어떻게 쌓였고, 누적 매출이 전체의 절반을 넘은 달은 언제인가?

WITH monthly AS (
  SELECT
    CAST(date_trunc('month', order_date) AS DATE) AS month,
    SUM(amount) AS revenue
  FROM project_name.dataset_name.orders
  WHERE amount IS NOT NULL
  GROUP BY month
),
cum AS (
  SELECT
    month,
    revenue,
    SUM(revenue) OVER (ORDER BY month) AS 누적매출,
    SUM(revenue) OVER () AS 전체매출
  FROM monthly
)
SELECT
  month,
  revenue AS 월매출,
  누적매출,
  ROUND(누적매출 * 100.0 / 전체매출, 1) AS 누적달성률_pct
FROM cum
ORDER BY month

Running query in 'duckdb'

month,월매출,누적매출,누적달성률_pct
2023-09-01,556000.00,556000.00,14.6
2023-10-01,775000.00,1331000.00,35.0
2023-11-01,839000.00,2170000.00,57.0
2023-12-01,526000.00,2696000.00,70.8
2024-01-01,850000.00,3546000.00,93.2
2024-02-01,260000.00,3806000.00,100.0


-> 누적 매출은 매달 전체의 14~22%p씩 쌓였고(2023년 12월만 13.8%p로 주춤), 2023년 11월에 2,170,000원(전체의 57.0%)으로 절반을 넘었다. 2024년 1월까지 93.2%가 쌓였으며, 2024년 2월이 6.8%(260,000원)뿐인 것은 데이터가 2월 11일까지만 있기 때문이라 해석에 주의해야 한다.

In [35]:
%%sql
-- [Q2] 전월 대비 매출 증감은 주문 수와 객단가 중 무엇 때문에 생겼는가?

WITH monthly AS (
  SELECT
    CAST(date_trunc('month', order_date) AS DATE) AS month,
    COUNT(*) AS 주문수,
    SUM(amount) AS revenue,
    ROUND(AVG(amount), 0) AS 객단가
  FROM project_name.dataset_name.orders
  WHERE amount IS NOT NULL
  GROUP BY month
),
with_prev AS (
  SELECT
    *,
    LAG(revenue) OVER (ORDER BY month) AS 전월매출,
    LAG(주문수) OVER (ORDER BY month) AS 전월주문수,
    LAG(객단가) OVER (ORDER BY month) AS 전월객단가
  FROM monthly
)
SELECT
  month,
  revenue AS 월매출,
  ROUND((revenue - 전월매출) * 100.0 / NULLIF(전월매출, 0), 1) AS 매출증감률_pct,
  주문수,
  ROUND((주문수 - 전월주문수) * 100.0 / NULLIF(전월주문수, 0), 1) AS 주문수증감률_pct,
  객단가,
  ROUND((객단가 - 전월객단가) * 100.0 / NULLIF(전월객단가, 0), 1) AS 객단가증감률_pct
FROM with_prev
ORDER BY month

Running query in 'duckdb'

month,월매출,매출증감률_pct,주문수,주문수증감률_pct,객단가,객단가증감률_pct
2023-09-01,556000.00,None,5,None,111200.0,None
2023-10-01,775000.00,39.4,5,0.0,155000.0,39.4
2023-11-01,839000.00,8.3,6,20.0,139833.0,-9.8
2023-12-01,526000.00,-37.3,5,-16.7,105200.0,-24.8
2024-01-01,850000.00,61.6,4,-20.0,212500.0,102.0
2024-02-01,260000.00,-69.4,2,-50.0,130000.0,-38.8


-> 5번의 전월 대비 변화 중 3번(2023년 10월·12월, 2024년 1월)은 객단가 증감률이, 2번(2023년 11월, 2024년 2월)은 주문 수 증감률이 더 컸다. 2024년 2월의 주문 수 감소(-50.0%)는 데이터가 2월 11일까지만 있어서 생긴 것이므로, 다 찬 달만 보면 매출은 주로 객단가에 좌우된다. 특히 2024년 1월은 주문이 5건에서 4건으로 줄었는데도 객단가가 102.0% 올라 매출이 61.6% 늘었다.

In [36]:
%%sql
-- [Q3] 카테고리 안에서 매출 1위 상품은 무엇이고, 한 상품에 매출이 쏠린 카테고리는 어디인가?

WITH product_rev AS (
  SELECT
    p.category,
    p.product_name,
    SUM(oi.quantity) AS 판매수량,
    SUM(oi.quantity * oi.unit_price) AS revenue
  FROM project_name.dataset_name.order_items AS oi
  JOIN project_name.dataset_name.products AS p
    ON oi.product_id = p.product_id
  GROUP BY p.category, p.product_name
)
SELECT
  category,
  product_name,
  판매수량,
  revenue AS 매출,
  DENSE_RANK() OVER (PARTITION BY category ORDER BY revenue DESC) AS 카테고리내_순위,
  ROUND(revenue * 100.0 / SUM(revenue) OVER (PARTITION BY category), 1) AS 카테고리내_비중_pct,
  RANK() OVER (ORDER BY revenue DESC) AS 전체_순위
FROM product_rev
ORDER BY SUM(revenue) OVER (PARTITION BY category) DESC, 카테고리내_순위

Running query in 'duckdb'

category,product_name,판매수량,매출,카테고리내_순위,카테고리내_비중_pct,전체_순위
Sandals,여름 샌들,5,551000.00,1,51.0,2
Sandals,슬리퍼 샌들,5,530000.00,2,49.0,3
Sneakers,캔버스 스니커즈,5,624000.00,1,63.1,1
Sneakers,클래식 스니커즈,5,365000.00,2,36.9,6
Boots,워커 부츠,4,410000.00,1,64.6,5
Boots,첼시 부츠,4,225000.00,2,35.4,9
Running,트레일 러너,4,361000.00,1,60.2,7
Running,에어러너,4,239000.00,2,39.8,8
Accessory,양말 세트,4,501000.00,1,100.0,4


-> 카테고리별 매출 1위 상품은 여름 샌들(Sandals), 캔버스 스니커즈(Sneakers), 워커 부츠(Boots), 트레일 러너(Running), 양말 세트(Accessory)다. 1위 상품의 비중은 Accessory가 100%(운동화 끈 판매 0)로 가장 쏠려 있고, Boots 64.6%, Sneakers 63.1%, Running 60.2% 순이다. Sandals만 51.0%로 두 상품이 고르게 팔렸다.

In [37]:
%%sql
-- [Q4] 재구매 고객은 직전 주문 후 평균 며칠 만에 다시 구매하는가?

WITH gaps AS (
  SELECT
    customer_id,
    order_date,
    date_diff(
      'day',
      LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date),
      order_date
    ) AS 구매간격_일
  FROM project_name.dataset_name.orders
  WHERE amount IS NOT NULL
)
SELECT
  customer_id,
  order_date,
  구매간격_일,
  ROUND(AVG(구매간격_일) OVER (PARTITION BY customer_id), 1) AS 고객평균_일,
  ROUND(AVG(구매간격_일) OVER (), 1) AS 전체평균_일
FROM gaps
WHERE 구매간격_일 IS NOT NULL
ORDER BY 고객평균_일, customer_id, order_date

Running query in 'duckdb'

customer_id,order_date,구매간격_일,고객평균_일,전체평균_일
C001,2023-09-11,9,36.3,65.8
C001,2023-10-23,42,36.3,65.8
C001,2023-12-20,58,36.3,65.8
C002,2023-10-01,26,40.0,65.8
C002,2023-11-24,54,40.0,65.8
C008,2023-12-09,51,51.0,65.8
C013,2024-01-22,54,54.0,65.8
C005,2023-12-03,69,69.0,65.8
C007,2024-01-03,83,83.0,65.8
C012,2024-02-11,84,84.0,65.8


-> 재구매 고객 10명(구매 고객 14명 중)의 재구매 13건은 평균 65.8일 만에 이뤄져 대략 2개월 주기로 다시 구매한다. 고객별 평균은 C001이 36.3일로 가장 짧고 C003이 122일로 가장 길다. 재구매 유도 메시지는 구매 후 약 2개월 무렵에 보내는 것이 적절해 보인다.

In [39]:
%%sql
-- [Q5] 고객을 총구매액 순으로 줄 세우면 상위 몇 % 고객이 매출의 몇 %를 만드는가?

WITH customer_rev AS (
  SELECT
    customer_id,
    SUM(amount) AS 총구매액
  FROM project_name.dataset_name.orders
  WHERE amount IS NOT NULL
  GROUP BY customer_id
),
ranked AS (
  SELECT
    *,
    ROW_NUMBER() OVER (ORDER BY 총구매액 DESC, customer_id) AS 순위,
    SUM(총구매액) OVER (ORDER BY 총구매액 DESC, customer_id) AS 누적구매액,
    SUM(총구매액) OVER () AS 전체구매액,
    COUNT(*) OVER () AS 전체고객수
  FROM customer_rev
)
SELECT
  순위,
  customer_id,
  총구매액,
  ROUND(순위 * 100.0 / 전체고객수, 1) AS 상위고객_pct,
  ROUND(누적구매액 * 100.0 / 전체구매액, 1) AS 누적매출비중_pct
FROM ranked
ORDER BY 순위

Running query in 'duckdb'

순위,customer_id,총구매액,상위고객_pct,누적매출비중_pct
1,C013,725000.00,7.1,19.0
2,C007,678000.00,14.3,36.9
3,C001,380000.00,21.4,46.8
4,C012,362000.00,28.6,56.4
5,C003,349000.00,35.7,65.5
6,C008,314000.00,42.9,73.8
7,C002,308000.00,50.0,81.9
8,C004,161000.00,57.1,86.1
9,C010,142000.00,64.3,89.8
10,C005,114000.00,71.4,92.8


-> 구매 고객 14명 중 상위 3명(21.4%)이 매출의 46.8%, 상위 7명(50%)이 81.9%를 만들어 매출이 소수 고객에게 몰려 있다.

## ⏭️ 다음 학습 예고 — SQL 실무: 사용자 행동 지표 분석

오늘 익힌 CTE와 윈도우 함수는 *비즈니스 분석의 무기*입니다. 다음 시간에는 이 무기를 총동원해 **코호트 리텐션**(가입 월별로 고객이 얼마나 남는가), **퍼널 분석**(어느 단계에서 이탈하는가), **RFM 세분화**(누가 우리의 핵심 고객인가)를 SQL로 구현합니다. 드디어 진짜 '데이터 분석가의 질문'에 답합니다.

오늘 여러분은 SQL을 _한 단계_ 끌어올렸습니다. CTE와 윈도우 함수는 실무 분석가가 매일 쓰는 도구입니다.

오늘도 한 걸음, 수고하셨습니다! 🎉

---

<sub>© 2026 모두의연구소(MODULABS). All rights reserved.<br>
기획·제작: 교육퍼실리테이터팀 이진영 (jy.lee@modulabs.co.kr)<br>
본 자료는 생성형 AI를 활용해 제작되었고, 제작자의 검수를 거쳐 완성되었습니다.<br>
무단 복제 및 배포를 금합니다.</sub>